In [ ]:
# ============================================================
# UsedCarAI — Reliability Audit
# Cell 1: Environment Setup
# ============================================================

from pathlib import Path
import os
import sys

# Discover CPE-306 when opened from the project or UsedcarAi directory.
# Set USEDCAR_PROJECT_DIR to use a different checkout location.
_candidates = (Path.cwd(), *Path.cwd().parents)
_default_project = next(
    (p for p in _candidates if (p / "dataset").is_dir() and (p / "app").is_dir()),
    Path("/home/regen/Works/CPE-306"),
)
PROJECT_DIR = Path(os.environ.get("USEDCAR_PROJECT_DIR", _default_project)).expanduser().resolve()
DATASET_DIR = Path(os.environ.get("DATASET_DIR", PROJECT_DIR / "dataset")).expanduser().resolve()
if not DATASET_DIR.is_dir():
    raise FileNotFoundError(f"Dataset directory not found: {DATASET_DIR}")
print("Python:", sys.executable)
print("Dataset:", DATASET_DIR)
print("Use the Python kernel from the project's ML environment.")

# Enable GPU acceleration before pandas or scikit-learn is imported.
_APP_IMPORT_DIR = str(PROJECT_DIR / "app")
if _APP_IMPORT_DIR not in sys.path:
    sys.path.insert(0, _APP_IMPORT_DIR)
from acceleration import enable_gpu, load_model
GPU_ENABLED = enable_gpu()

from pathlib import Path
import sys
import warnings

PROJECT_ROOT = PROJECT_DIR
APP_DIR = PROJECT_ROOT / "app"
DATA_DIR = DATASET_DIR / "data"
MODEL_DIR = DATASET_DIR / "models"
NOTEBOOK_DIR = PROJECT_DIR / "UsedcarAi"

# ให้ import utils.py จาก app/
if str(APP_DIR) not in sys.path:
    sys.path.insert(0, str(APP_DIR))

print("=== RELIABILITY AUDIT — ENVIRONMENT SETUP ===")
print("Project root :", PROJECT_ROOT)
print("App dir      :", APP_DIR)
print("Data dir     :", DATA_DIR)
print("Model dir    :", MODEL_DIR)
print("Notebook dir :", NOTEBOOK_DIR)

# ตรวจโครงสร้างสำคัญก่อนทำงานต่อ
required_paths = [
    PROJECT_ROOT,
    APP_DIR,
    DATA_DIR,
    MODEL_DIR,
    APP_DIR / "utils.py",
    APP_DIR / "prediction_app.py",
]

missing = [p for p in required_paths if not p.exists()]

if missing:
    print("\nERROR: ไม่พบไฟล์/โฟลเดอร์:")
    for p in missing:
        print(" -", p)
    raise FileNotFoundError(
        "Project structure ไม่ครบ — หยุดก่อนเพื่อป้องกัน audit ผิดโปรเจกต์"
    )

print("\nPASS: Project structure OK")
print("พร้อมสำหรับ Cell 2")

In [ ]:
# ============================================================
# UsedCarAI — Reliability Audit
# Cell 2: Project Artifact Inventory
# ============================================================

print("=== PROJECT ARTIFACT INVENTORY ===")

# นามสกุลที่เกี่ยวข้องกับ data / model
interesting_ext = {
    ".csv", ".parquet", ".pkl", ".pickle",
    ".joblib", ".json", ".yaml", ".yml"
}

def show_files(folder, title):
    print(f"\n=== {title} ===")

    if not folder.exists():
        print("Folder not found:", folder)
        return []

    files = sorted([
        p for p in folder.rglob("*")
        if p.is_file() and p.suffix.lower() in interesting_ext
    ])

    if not files:
        print("No relevant files found.")
        return []

    for p in files:
        try:
            size_mb = p.stat().st_size / (1024 ** 2)
            relative = p.relative_to(PROJECT_ROOT)
            print(f"{relative}  [{size_mb:.2f} MB]")
        except Exception:
            print(p)

    return files


data_files = show_files(
    DATA_DIR,
    "DATA FILES"
)

model_files = show_files(
    MODEL_DIR,
    "MODEL FILES"
)

print("\n=== APP PYTHON FILES ===")

app_py_files = sorted(APP_DIR.glob("*.py"))

for p in app_py_files:
    print(p.relative_to(PROJECT_ROOT))


print("\n=== INVENTORY SUMMARY ===")
print("Data artifacts :", len(data_files))
print("Model artifacts:", len(model_files))
print("Python files   :", len(app_py_files))


if len(data_files) == 0:
    print("\nWARNING: ไม่พบ data artifact")

if len(model_files) == 0:
    print("\nWARNING: ไม่พบ model artifact")


print("\nPASS: Artifact inventory complete")
print("ยังไม่มีการแก้ไข production files")
print("พร้อมสำหรับ Cell 3")

In [ ]:
# ============================================================
# UsedCarAI — Reliability Audit
# Cell 3: Load Production Artifacts (READ ONLY)
# ============================================================

import json
import joblib
import pandas as pd
from pathlib import Path

print("=== LOAD PRODUCTION ARTIFACTS ===")

# ------------------------------------------------------------
# Paths
# ------------------------------------------------------------

CLEANED_PATH = DATA_DIR / "cleaned" / "one2car_cleaned.csv"

TRAIN_PATH = DATA_DIR / "processed" / "train.csv"
VALIDATION_PATH = DATA_DIR / "processed" / "validation.csv"
TEST_PATH = DATA_DIR / "processed" / "test.csv"

RF_MODEL_PATH = (
    MODEL_DIR
    / "random_forest"
    / "random_forest_pipeline.joblib"
)

RF_META_PATH = (
    MODEL_DIR
    / "random_forest"
    / "random_forest_metadata.json"
)

LR_MODEL_PATH = (
    MODEL_DIR
    / "linear_regression"
    / "linear_regression_pipeline.joblib"
)

LR_META_PATH = (
    MODEL_DIR
    / "linear_regression"
    / "linear_regression_metadata.json"
)

HYBRID_META_PATH = (
    MODEL_DIR
    / "hybrid"
    / "hybrid_metadata.json"
)

HYBRID_MARKET_PATH = (
    MODEL_DIR
    / "hybrid"
    / "market_reference_train_only.csv"
)


# ------------------------------------------------------------
# Required artifact check
# ------------------------------------------------------------

required = [
    CLEANED_PATH,
    TRAIN_PATH,
    VALIDATION_PATH,
    TEST_PATH,
    RF_MODEL_PATH,
    RF_META_PATH,
    LR_MODEL_PATH,
    LR_META_PATH,
    HYBRID_META_PATH,
    HYBRID_MARKET_PATH,
]

missing = [p for p in required if not p.exists()]

if missing:
    print("\nERROR: Missing artifacts:")
    for p in missing:
        print(" -", p)

    raise FileNotFoundError(
        "Required production artifacts are incomplete. Run 01_data_cleaning.ipynb "
        "to generate the cleaned dataset, then 02–04 for model artifacts."
    )

print("PASS: All required artifacts found")


# ------------------------------------------------------------
# Load datasets
# ------------------------------------------------------------

print("\n=== LOADING DATASETS ===")

cleaned = pd.read_csv(CLEANED_PATH)
train_df = pd.read_csv(TRAIN_PATH)
validation_df = pd.read_csv(VALIDATION_PATH)
test_df = pd.read_csv(TEST_PATH)

print("cleaned   :", cleaned.shape)
print("train     :", train_df.shape)
print("validation:", validation_df.shape)
print("test      :", test_df.shape)


# ------------------------------------------------------------
# Load production models
# ------------------------------------------------------------

print("\n=== LOADING PRODUCTION MODELS ===")

rf_model = load_model(RF_MODEL_PATH)
lr_model = load_model(LR_MODEL_PATH)

print("Random Forest:")
print(" ", type(rf_model))

print("Linear Regression:")
print(" ", type(lr_model))


# ------------------------------------------------------------
# Load metadata
# ------------------------------------------------------------

print("\n=== LOADING METADATA ===")

with open(RF_META_PATH, "r", encoding="utf-8") as f:
    rf_metadata = json.load(f)

with open(LR_META_PATH, "r", encoding="utf-8") as f:
    lr_metadata = json.load(f)

with open(HYBRID_META_PATH, "r", encoding="utf-8") as f:
    hybrid_metadata = json.load(f)

market_reference = pd.read_csv(HYBRID_MARKET_PATH)

print("RF metadata keys:")
print(list(rf_metadata.keys()))

print("\nLR metadata keys:")
print(list(lr_metadata.keys()))

print("\nHybrid metadata keys:")
print(list(hybrid_metadata.keys()))

print("\nMarket reference:")
print(market_reference.shape)


# ------------------------------------------------------------
# Basic integrity checks
# ------------------------------------------------------------

print("\n=== BASIC INTEGRITY CHECK ===")

print("Cleaned rows     :", len(cleaned))
print("Train rows       :", len(train_df))
print("Validation rows  :", len(validation_df))
print("Test rows        :", len(test_df))

total_split = (
    len(train_df)
    + len(validation_df)
    + len(test_df)
)

print("Total split rows :", total_split)

if total_split == len(cleaned):
    print("PASS: train + validation + test == cleaned")
else:
    print(
        "INFO: split total differs from cleaned:",
        total_split,
        "vs",
        len(cleaned)
    )


# ------------------------------------------------------------
# Model feature information
# ------------------------------------------------------------

print("\n=== MODEL FEATURE INFO ===")

for name, model in [
    ("Random Forest", rf_model),
    ("Linear Regression", lr_model),
]:
    print(f"\n{name}")

    if hasattr(model, "feature_names_in_"):
        print(
            "feature_names_in_:",
            list(model.feature_names_in_)
        )
    else:
        print(
            "feature_names_in_: not exposed "
            "at pipeline level"
        )

    if hasattr(model, "n_features_in_"):
        print(
            "n_features_in_:",
            model.n_features_in_
        )


# ------------------------------------------------------------
# Safety confirmation
# ------------------------------------------------------------

print("\n=== CELL 3 COMPLETE ===")
print("Production artifacts loaded successfully.")
print("No production files modified.")
print("No model retraining performed.")
print("Ready for Cell 4.")

In [ ]:
# ============================================================
# UsedCarAI — Reliability Audit
# Cell 4: Build Reproducible Audit Base
# ============================================================

import numpy as np
import pandas as pd

print("=== BUILD AUDIT BASE ===")

# ------------------------------------------------------------
# 1. Copy test set
# ------------------------------------------------------------

audit = test_df.copy()

print("Original test rows:", len(audit))
print("Columns:", len(audit.columns))


# ------------------------------------------------------------
# 2. Required columns
# ------------------------------------------------------------

required_cols = [
    "listing_id",
    "price",
    "brand",
    "model",
    "sub_model",
    "model_year",
    "mileage",
]

missing_cols = [
    c for c in required_cols
    if c not in audit.columns
]

if missing_cols:
    raise KeyError(
        "Missing required columns: "
        + ", ".join(missing_cols)
    )

print("PASS: required columns found")


# ------------------------------------------------------------
# 3. listing_id integrity
# ------------------------------------------------------------

print("\n=== LISTING ID CHECK ===")

print("Rows:", len(audit))
print("Non-null listing_id:", audit["listing_id"].notna().sum())
print("Unique listing_id:", audit["listing_id"].nunique())
print("Duplicate listing_id:", audit["listing_id"].duplicated().sum())

if audit["listing_id"].isna().any():
    raise ValueError("listing_id contains missing values")

if audit["listing_id"].duplicated().any():
    raise ValueError("listing_id contains duplicates")

print("PASS: listing_id is unique and complete")


# ------------------------------------------------------------
# 4. Target integrity
# ------------------------------------------------------------

print("\n=== TARGET CHECK ===")

audit["price"] = pd.to_numeric(
    audit["price"],
    errors="coerce"
)

invalid_price = (
    audit["price"].isna()
    | ~np.isfinite(audit["price"])
    | (audit["price"] <= 0)
)

print("Invalid target rows:", int(invalid_price.sum()))

if invalid_price.any():
    raise ValueError(
        "Audit test set contains invalid price values"
    )

print("PASS: target price valid")


# ------------------------------------------------------------
# 5. Model feature compatibility
# ------------------------------------------------------------

print("\n=== MODEL FEATURE COMPATIBILITY ===")

rf_features = list(rf_model.feature_names_in_)
lr_features = list(lr_model.feature_names_in_)

print("RF features:", len(rf_features))
print("LR features:", len(lr_features))

if rf_features != lr_features:
    raise ValueError(
        "RF and LR feature definitions are different"
    )

MODEL_FEATURES = rf_features

missing_features = [
    c for c in MODEL_FEATURES
    if c not in audit.columns
]

print("Required model features:", MODEL_FEATURES)
print("Missing from test:", missing_features)

if missing_features:
    raise KeyError(
        "Test set missing model features: "
        + ", ".join(missing_features)
    )

print("PASS: test set compatible with production models")


# ------------------------------------------------------------
# 6. Extreme flags
# ------------------------------------------------------------

print("\n=== EXTREME FLAG CHECK ===")

for col in [
    "price_extreme_flag",
    "mileage_extreme_flag",
]:
    if col in audit.columns:
        print(f"\n{col}")
        print(
            audit[col]
            .value_counts(dropna=False)
            .to_string()
        )
    else:
        print(f"{col}: not present")


# ------------------------------------------------------------
# 7. Create conservative audit subset
# ------------------------------------------------------------

audit_mask = pd.Series(
    True,
    index=audit.index
)

if "price_extreme_flag" in audit.columns:
    audit_mask &= ~audit["price_extreme_flag"].fillna(False)

if "mileage_extreme_flag" in audit.columns:
    audit_mask &= ~audit["mileage_extreme_flag"].fillna(False)

audit_core = (
    audit.loc[audit_mask]
    .copy()
    .reset_index(drop=True)
)

print("\n=== AUDIT CORE ===")
print("Full test rows :", len(audit))
print("Core rows      :", len(audit_core))
print("Excluded rows  :", len(audit) - len(audit_core))


# ------------------------------------------------------------
# 8. Coverage information
# ------------------------------------------------------------

train_brand_model = set(
    zip(
        train_df["brand"].astype(str),
        train_df["model"].astype(str),
    )
)

audit_pairs = list(
    zip(
        audit_core["brand"].astype(str),
        audit_core["model"].astype(str),
    )
)

seen_brand_model = pd.Series(
    [
        pair in train_brand_model
        for pair in audit_pairs
    ]
)

print("\n=== TRAIN COVERAGE ===")

print(
    "Brand+Model seen in train:",
    int(seen_brand_model.sum()),
    "/",
    len(audit_core),
)

print(
    "Brand+Model unseen in train:",
    int((~seen_brand_model).sum()),
)


# ------------------------------------------------------------
# 9. Summary
# ------------------------------------------------------------

print("\n=== AUDIT BASE SUMMARY ===")

print(
    audit_core[
        [
            "price",
            "model_year",
            "mileage",
        ]
    ].describe()
)

print("\nBrands:", audit_core["brand"].nunique())
print("Models:", audit_core["model"].nunique())

print("\n=== CELL 4 COMPLETE ===")
print("audit      :", audit.shape)
print("audit_core :", audit_core.shape)
print("Production models untouched.")
print("Ready for Cell 5.")

In [ ]:
# ============================================================
# UsedCarAI — Reliability Audit
# Cell 5: Production Predictions & Error Baseline
# ============================================================

import numpy as np
import pandas as pd
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score,
)

print("=== PRODUCTION MODEL BASELINE ===")

# ------------------------------------------------------------
# 1. Prepare audit prediction table
# ------------------------------------------------------------

audit_pred = audit_core.copy()

X_audit = audit_pred[MODEL_FEATURES].copy()
y_true = audit_pred["price"].astype(float).to_numpy()

print("Audit rows :", len(audit_pred))
print("Features   :", X_audit.shape[1])


# ------------------------------------------------------------
# 2. Production predictions
# ------------------------------------------------------------

print("\n=== RUNNING PRODUCTION MODELS ===")

audit_pred["rf_prediction"] = rf_model.predict(X_audit)
audit_pred["lr_prediction"] = lr_model.predict(X_audit)

print("Random Forest prediction complete.")
print("Linear Regression prediction complete.")


# ------------------------------------------------------------
# 3. Prediction sanity checks
# ------------------------------------------------------------

for col in ["rf_prediction", "lr_prediction"]:

    audit_pred[col] = pd.to_numeric(
        audit_pred[col],
        errors="coerce"
    )

    invalid = (
        audit_pred[col].isna()
        | ~np.isfinite(audit_pred[col])
    )

    print(
        f"{col} invalid:",
        int(invalid.sum())
    )

    if invalid.any():
        raise ValueError(
            f"{col} contains invalid predictions"
        )

print("PASS: predictions are finite")


# ------------------------------------------------------------
# 4. Error columns
# ------------------------------------------------------------

EPS = 1.0

for prefix in ["rf", "lr"]:

    pred = audit_pred[f"{prefix}_prediction"]

    audit_pred[f"{prefix}_error"] = (
        pred - audit_pred["price"]
    )

    audit_pred[f"{prefix}_abs_error"] = (
        audit_pred[f"{prefix}_error"].abs()
    )

    audit_pred[f"{prefix}_ape"] = (
        audit_pred[f"{prefix}_abs_error"]
        / audit_pred["price"].clip(lower=EPS)
    )

    # Symmetric percentage error
    audit_pred[f"{prefix}_sape"] = (
        2.0 * audit_pred[f"{prefix}_abs_error"]
        / (
            audit_pred["price"].abs()
            + pred.abs()
        ).clip(lower=EPS)
    )


# ------------------------------------------------------------
# 5. Metric function
# ------------------------------------------------------------

def regression_metrics(y, pred):

    y = np.asarray(y, dtype=float)
    pred = np.asarray(pred, dtype=float)

    abs_error = np.abs(y - pred)

    ape = (
        abs_error
        / np.maximum(np.abs(y), 1.0)
    )

    sape = (
        2.0 * abs_error
        / np.maximum(
            np.abs(y) + np.abs(pred),
            1.0
        )
    )

    return {
        "MAE": mean_absolute_error(y, pred),
        "RMSE": np.sqrt(
            mean_squared_error(y, pred)
        ),
        "Median_AE": np.median(abs_error),
        "MAPE_percent": np.mean(ape) * 100,
        "Median_APE_percent": np.median(ape) * 100,
        "sMAPE_percent": np.mean(sape) * 100,
        "R2": r2_score(y, pred),
    }


# ------------------------------------------------------------
# 6. Calculate baseline metrics
# ------------------------------------------------------------

rf_metrics = regression_metrics(
    audit_pred["price"],
    audit_pred["rf_prediction"],
)

lr_metrics = regression_metrics(
    audit_pred["price"],
    audit_pred["lr_prediction"],
)

baseline_metrics = pd.DataFrame(
    [
        {
            "model": "Random Forest",
            **rf_metrics
        },
        {
            "model": "Linear Regression",
            **lr_metrics
        },
    ]
)

print("\n=== AUDIT BASELINE METRICS ===")

display(
    baseline_metrics.round(
        {
            "MAE": 0,
            "RMSE": 0,
            "Median_AE": 0,
            "MAPE_percent": 2,
            "Median_APE_percent": 2,
            "sMAPE_percent": 2,
            "R2": 4,
        }
    )
)


# ------------------------------------------------------------
# 7. RF vs LR disagreement
# ------------------------------------------------------------

audit_pred["model_disagreement"] = (
    np.abs(
        audit_pred["rf_prediction"]
        - audit_pred["lr_prediction"]
    )
    /
    np.maximum(
        np.maximum(
            np.abs(audit_pred["rf_prediction"]),
            np.abs(audit_pred["lr_prediction"]),
        ),
        1.0,
    )
).clip(0, 1)


print("\n=== MODEL DISAGREEMENT ===")

print(
    audit_pred["model_disagreement"]
    .describe(
        percentiles=[
            .25, .50, .75,
            .90, .95, .99
        ]
    )
)


# ------------------------------------------------------------
# 8. Error percentiles
# ------------------------------------------------------------

print("\n=== ABSOLUTE ERROR PERCENTILES ===")

error_percentiles = pd.DataFrame(
    {
        "RF": audit_pred["rf_abs_error"].quantile(
            [.25, .50, .75, .90, .95, .99]
        ),
        "LR": audit_pred["lr_abs_error"].quantile(
            [.25, .50, .75, .90, .95, .99]
        ),
    }
)

display(error_percentiles.round(0))


# ------------------------------------------------------------
# 9. Check against stored production metadata
# ------------------------------------------------------------

print("\n=== STORED TEST METRICS ===")

print("RF metadata test_metrics:")
print(
    rf_metadata.get(
        "test_metrics",
        "N/A"
    )
)

print("\nLR metadata test_metrics:")
print(
    lr_metadata.get(
        "test_metrics",
        "N/A"
    )
)


# ------------------------------------------------------------
# 10. Final integrity
# ------------------------------------------------------------

assert len(audit_pred) == len(audit_core)

assert (
    audit_pred["listing_id"].nunique()
    == len(audit_pred)
)

print("\n=== CELL 5 COMPLETE ===")
print("Rows evaluated :", len(audit_pred))
print("Production models untouched.")
print("Predictions stored in: audit_pred")
print("Ready for Cell 6.")

In [ ]:
# ============================================================
# UsedCarAI — Reliability Audit
# Cell 5.1: Reconcile Metrics Against Full Production Test Set
# ============================================================

import numpy as np
import pandas as pd
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score,
)

print("=== FULL TEST METRIC RECONCILIATION ===")

# ------------------------------------------------------------
# 1. Use ORIGINAL full test set
# ------------------------------------------------------------

full_test = test_df.copy()

X_full = full_test[MODEL_FEATURES].copy()
y_full = full_test["price"].astype(float).to_numpy()

print("Full test rows:", len(full_test))
print("Features:", X_full.shape[1])


# ------------------------------------------------------------
# 2. Run production models
# ------------------------------------------------------------

rf_full_pred = rf_model.predict(X_full)
lr_full_pred = lr_model.predict(X_full)

print("Predictions complete.")


# ------------------------------------------------------------
# 3. Metric calculation
# ------------------------------------------------------------

def reconcile_metrics(y, pred):

    y = np.asarray(y, dtype=float)
    pred = np.asarray(pred, dtype=float)

    abs_error = np.abs(y - pred)

    sape = (
        2.0 * abs_error
        /
        np.maximum(
            np.abs(y) + np.abs(pred),
            1.0
        )
    )

    return {
        "MAE_THB":
            mean_absolute_error(y, pred),

        "RMSE_THB":
            np.sqrt(
                mean_squared_error(y, pred)
            ),

        "R2":
            r2_score(y, pred),

        "sMAPE_percent":
            np.mean(sape) * 100,

        "MedianAE_THB":
            np.median(abs_error),
    }


rf_full_metrics = reconcile_metrics(
    y_full,
    rf_full_pred
)

lr_full_metrics = reconcile_metrics(
    y_full,
    lr_full_pred
)


# ------------------------------------------------------------
# 4. Metadata
# ------------------------------------------------------------

rf_stored = rf_metadata.get(
    "test_metrics",
    {}
)

lr_stored = lr_metadata.get(
    "test_metrics",
    {}
)


# ------------------------------------------------------------
# 5. Comparison table
# ------------------------------------------------------------

rows = []

for model_name, current, stored in [
    (
        "Random Forest",
        rf_full_metrics,
        rf_stored,
    ),
    (
        "Linear Regression",
        lr_full_metrics,
        lr_stored,
    ),
]:

    for metric in [
        "MAE_THB",
        "RMSE_THB",
        "R2",
        "sMAPE_percent",
        "MedianAE_THB",
    ]:

        current_value = current.get(
            metric,
            np.nan
        )

        stored_value = stored.get(
            metric,
            np.nan
        )

        rows.append(
            {
                "model": model_name,
                "metric": metric,
                "current_full_test": current_value,
                "stored_metadata": stored_value,
                "difference":
                    current_value - stored_value
                    if (
                        pd.notna(current_value)
                        and pd.notna(stored_value)
                    )
                    else np.nan
            }
        )


reconciliation = pd.DataFrame(rows)

print("\n=== CURRENT vs STORED ===")

display(reconciliation)


# ------------------------------------------------------------
# 6. Target variance comparison
# ------------------------------------------------------------

print("\n=== TARGET DISTRIBUTION COMPARISON ===")

target_compare = pd.DataFrame(
    {
        "full_test": full_test["price"].describe(),
        "audit_core": audit_core["price"].describe(),
    }
)

display(target_compare)


print("\nTarget variance:")
print(
    "Full test :",
    float(full_test["price"].var())
)

print(
    "Audit core:",
    float(audit_core["price"].var())
)


# ------------------------------------------------------------
# 7. R2 explanation check
# ------------------------------------------------------------

print("\n=== R2 CHECK ===")

print(
    "RF full-test R2 :",
    round(rf_full_metrics["R2"], 6)
)

print(
    "RF core R2      :",
    round(rf_metrics["R2"], 6)
)

print(
    "LR full-test R2 :",
    round(lr_full_metrics["R2"], 6)
)

print(
    "LR core R2      :",
    round(lr_metrics["R2"], 6)
)


# ------------------------------------------------------------
# 8. Automatic reconciliation status
# ------------------------------------------------------------

def close_enough(a, b, rtol=1e-6, atol=1e-6):

    if a is None or b is None:
        return False

    try:
        return np.isclose(
            float(a),
            float(b),
            rtol=rtol,
            atol=atol,
        )
    except Exception:
        return False


rf_checks = {
    k: close_enough(
        rf_full_metrics.get(k),
        rf_stored.get(k)
    )
    for k in rf_full_metrics
}

lr_checks = {
    k: close_enough(
        lr_full_metrics.get(k),
        lr_stored.get(k)
    )
    for k in lr_full_metrics
}


print("\n=== RECONCILIATION STATUS ===")

print("RF:")
for k, v in rf_checks.items():
    print(f"  {k}: {'MATCH' if v else 'DIFFER'}")

print("LR:")
for k, v in lr_checks.items():
    print(f"  {k}: {'MATCH' if v else 'DIFFER'}")


# ------------------------------------------------------------
# 9. Finish
# ------------------------------------------------------------

print("\n=== CELL 5.1 COMPLETE ===")
print("No production files modified.")
print("No models retrained.")

In [ ]:
# ============================================================
# CELL 6 — RELIABILITY CALIBRATION ANALYSIS
# Goal:
#   ตรวจว่า reliability score/level ที่ production logic ให้
#   สัมพันธ์กับ prediction error จริงหรือไม่
#
# IMPORTANT:
#   - ไม่แก้ production model
#   - ไม่ retrain model
#   - ไม่แก้ utils.py
#   - วิเคราะห์จาก audit_pred เท่านั้น
# ============================================================

import numpy as np
import pandas as pd

print("=== RELIABILITY CALIBRATION ANALYSIS ===")

# ------------------------------------------------------------
# 1. Precheck
# ------------------------------------------------------------
if "audit_pred" not in globals():
    raise RuntimeError(
        "ไม่พบ audit_pred — กรุณารัน Cell 5 ก่อน"
    )

cal = audit_pred.copy()

required = [
    "price",
    "rf_prediction",
    "lr_prediction",
]

missing = [c for c in required if c not in cal.columns]

if missing:
    raise RuntimeError(
        "audit_pred ขาด columns: " + ", ".join(missing)
    )

print("Rows:", len(cal))


# ------------------------------------------------------------
# 2. Find reliability columns already produced by audit
# ------------------------------------------------------------
score_candidates = [
    "reliability_score",
    "score",
]

level_candidates = [
    "reliability_level",
    "level",
]

score_col = next(
    (c for c in score_candidates if c in cal.columns),
    None
)

level_col = next(
    (c for c in level_candidates if c in cal.columns),
    None
)

print("Reliability score column:", score_col)
print("Reliability level column:", level_col)


# ------------------------------------------------------------
# 3. Actual prediction errors
# ------------------------------------------------------------
cal["rf_abs_error"] = np.abs(
    cal["price"] - cal["rf_prediction"]
)

cal["lr_abs_error"] = np.abs(
    cal["price"] - cal["lr_prediction"]
)

# ใช้ RF เป็น production ML baseline หลัก
cal["abs_error"] = cal["rf_abs_error"]

# Percentage error
denom = cal["price"].replace(0, np.nan)

cal["ape_percent"] = (
    cal["abs_error"] / denom * 100
)


# ------------------------------------------------------------
# 4. Basic error summary
# ------------------------------------------------------------
print("\n=== ACTUAL ERROR SUMMARY ===")

print(
    cal[
        [
            "abs_error",
            "ape_percent",
        ]
    ].describe(
        percentiles=[.25, .50, .75, .90, .95, .99]
    )
)


# ------------------------------------------------------------
# 5. Reliability score distribution
# ------------------------------------------------------------
if score_col is not None:

    cal[score_col] = pd.to_numeric(
        cal[score_col],
        errors="coerce"
    )

    print("\n=== RELIABILITY SCORE DISTRIBUTION ===")

    print(
        cal[score_col].describe(
            percentiles=[
                .10, .25, .50,
                .75, .90, .95, .99
            ]
        )
    )


# ------------------------------------------------------------
# 6. Reliability level distribution
# ------------------------------------------------------------
if level_col is not None:

    print("\n=== RELIABILITY LEVEL DISTRIBUTION ===")

    print(
        cal[level_col]
        .value_counts(dropna=False)
    )


# ------------------------------------------------------------
# 7. Error by reliability level
# ------------------------------------------------------------
if level_col is not None:

    print("\n=== ERROR BY RELIABILITY LEVEL ===")

    level_summary = (
        cal
        .groupby(level_col, dropna=False)
        .agg(
            cars=("price", "size"),

            MAE=(
                "abs_error",
                "mean"
            ),

            Median_AE=(
                "abs_error",
                "median"
            ),

            Mean_APE_percent=(
                "ape_percent",
                "mean"
            ),

            Median_APE_percent=(
                "ape_percent",
                "median"
            ),
        )
        .reset_index()
    )

    display(level_summary)


# ------------------------------------------------------------
# 8. Error by score band
# ------------------------------------------------------------
if score_col is not None:

    cal["score_band"] = pd.cut(
        cal[score_col],
        bins=[
            -np.inf,
            39,
            59,
            79,
            np.inf,
        ],
        labels=[
            "0-39",
            "40-59",
            "60-79",
            "80-100",
        ],
    )

    print("\n=== ERROR BY SCORE BAND ===")

    score_summary = (
        cal
        .groupby(
            "score_band",
            observed=False
        )
        .agg(
            cars=("price", "size"),

            score_mean=(
                score_col,
                "mean"
            ),

            MAE=(
                "abs_error",
                "mean"
            ),

            Median_AE=(
                "abs_error",
                "median"
            ),

            Mean_APE_percent=(
                "ape_percent",
                "mean"
            ),

            Median_APE_percent=(
                "ape_percent",
                "median"
            ),
        )
        .reset_index()
    )

    display(score_summary)


# ------------------------------------------------------------
# 9. Score vs actual error correlation
# ------------------------------------------------------------
if score_col is not None:

    valid = (
        cal[
            [score_col, "abs_error"]
        ]
        .replace(
            [np.inf, -np.inf],
            np.nan
        )
        .dropna()
    )

    if len(valid) > 2:

        pearson_corr = valid[
            score_col
        ].corr(
            valid["abs_error"],
            method="pearson"
        )

        spearman_corr = valid[
            score_col
        ].corr(
            valid["abs_error"],
            method="spearman"
        )

        print("\n=== SCORE vs ACTUAL ERROR ===")

        print(
            "Pearson:",
            round(float(pearson_corr), 4)
        )

        print(
            "Spearman:",
            round(float(spearman_corr), 4)
        )

        print(
            "\nExpected direction: NEGATIVE"
        )
        print(
            "Higher reliability score should generally "
            "correspond to lower actual error."
        )


# ------------------------------------------------------------
# 10. High-confidence but large-error cases
# ------------------------------------------------------------
if score_col is not None:

    error_p90 = cal[
        "abs_error"
    ].quantile(.90)

    overconfident = cal[
        (cal[score_col] >= 80)
        &
        (cal["abs_error"] >= error_p90)
    ].copy()

    print(
        "\n=== HIGH CONFIDENCE / HIGH ERROR CASES ==="
    )

    print(
        "90th percentile absolute error:",
        round(float(error_p90), 2)
    )

    print(
        "High-score rows with >= P90 error:",
        len(overconfident),
        "/",
        len(cal)
    )

    show_cols = [
        c for c in [
            "listing_id",
            "brand",
            "model",
            "sub_model",
            "model_year",
            "price",
            "rf_prediction",
            score_col,
            level_col,
            "abs_error",
            "ape_percent",
        ]
        if c is not None and c in overconfident.columns
    ]

    if len(overconfident) > 0:
        display(
            overconfident[
                show_cols
            ]
            .sort_values(
                "abs_error",
                ascending=False
            )
            .head(20)
        )


# ------------------------------------------------------------
# 11. Final status
# ------------------------------------------------------------
print("\n=== CELL 6 COMPLETE ===")
print("Rows analyzed:", len(cal))
print("Production models untouched.")
print("No retraining performed.")
print("Reliability calibration analysis complete.")

In [ ]:
# ============================================================
# CELL 6A — INSPECT PRODUCTION RELIABILITY FUNCTION
# ============================================================

import sys
import inspect
from pathlib import Path

APP_DIR = PROJECT_DIR / "app"

if str(APP_DIR) not in sys.path:
    sys.path.insert(0, str(APP_DIR))

import utils

print("=== PRODUCTION RELIABILITY FUNCTION INSPECTION ===")

func = utils.evaluate_prediction_reliability

print("\nFunction:")
print(func)

print("\nSignature:")
print(inspect.signature(func))

print("\nSource:")
print("-" * 70)

try:
    print(inspect.getsource(func))
except Exception as e:
    print("Could not read source:", repr(e))

print("-" * 70)

print("\n=== RELATED RELIABILITY FUNCTIONS ===")

for name in dir(utils):
    low = name.lower()

    if (
        "reliab" in low
        or "confidence" in low
        or "score" in low
    ):
        obj = getattr(utils, name)

        if callable(obj):
            try:
                sig = inspect.signature(obj)
            except Exception:
                sig = "signature unavailable"

            print(f"{name}{sig}")

print("\n=== CELL 6A COMPLETE ===")
print("No production files modified.")
print("No models retrained.")

In [ ]:
# ============================================================
# CELL 6B — BUILD PRODUCTION RELIABILITY FOR AUDIT SET
# ============================================================

import numpy as np
import pandas as pd
import time

from utils import (
    prepare_input,
    predict_all,
    find_comparable,
    evaluate_prediction_reliability,
)

print("=== BUILDING PRODUCTION RELIABILITY DATA ===")

if "audit_pred" not in globals():
    raise RuntimeError("ไม่พบ audit_pred — กรุณารัน Cell 5 ก่อน")

cal = audit_pred.copy()

print("Audit rows:", len(cal))
print("Columns:", len(cal.columns))


# ------------------------------------------------------------
# 1. Detect existing prediction columns
# ------------------------------------------------------------

rf_candidates = [
    "rf_prediction",
    "rf_price",
]

lr_candidates = [
    "lr_prediction",
    "lr_price",
]

hybrid_candidates = [
    "hybrid_prediction",
    "hybrid_price",
]

rf_col = next(
    (c for c in rf_candidates if c in cal.columns),
    None
)

lr_col = next(
    (c for c in lr_candidates if c in cal.columns),
    None
)

hybrid_col = next(
    (c for c in hybrid_candidates if c in cal.columns),
    None
)

print("\nPrediction columns:")
print("RF     :", rf_col)
print("LR     :", lr_col)
print("Hybrid :", hybrid_col)

if rf_col is None or lr_col is None:
    raise RuntimeError(
        "ไม่พบ RF/LR prediction columns ใน audit_pred"
    )


# ------------------------------------------------------------
# 2. Containers
# ------------------------------------------------------------

reliability_scores = []
reliability_levels = []
reliability_warnings = []
model_disagreements = []

comparable_counts = []
similarities = []

failed_rows = []


# ------------------------------------------------------------
# 3. Run production comparable + reliability logic
# ------------------------------------------------------------

print("\n=== RUNNING RELIABILITY PIPELINE ===")
print("อาจใช้เวลาสักครู่ เพราะมี", len(cal), "rows")

start = time.time()

for i, (idx, row) in enumerate(cal.iterrows(), start=1):

    try:

        # ----------------------------------------------
        # Build production query
        # ----------------------------------------------
        query = prepare_input(
            brand=row["brand"],
            model=row["model"],
            sub_model=row.get("sub_model"),
            model_year=row["model_year"],
            mileage=row["mileage"],
            fuel_type=row.get("fuel_type"),
            transmission=row.get("transmission"),
            body_type=row.get("body_type"),
            color=row.get("color"),
            province=row.get("province"),
            location=row.get("location"),
            seller_type=row.get("seller_type"),
            engine_size=row.get("engine_size"),
            number_of_seats=row.get("number_of_seats"),
        )

        # ----------------------------------------------
        # Production comparable search
        # ----------------------------------------------
        comparable = find_comparable(query)

        if comparable is None:
            comp_count = 0
            similarity = None

        else:
            comp_count = len(comparable)

            if (
                comp_count > 0
                and "similarity" in comparable.columns
            ):
                similarity = pd.to_numeric(
                    comparable["similarity"],
                    errors="coerce"
                ).max()

                if pd.isna(similarity):
                    similarity = None

            else:
                similarity = None

        # ----------------------------------------------
        # Predictions already generated in Cell 5
        # ----------------------------------------------
        rf_price = float(row[rf_col])
        lr_price = float(row[lr_col])

        # Hybrid is not used internally by current
        # reliability function, but supply a valid value.
        if (
            hybrid_col is not None
            and pd.notna(row[hybrid_col])
        ):
            hybrid_price = float(row[hybrid_col])
        else:
            hybrid_price = (
                rf_price + lr_price
            ) / 2.0

        # ----------------------------------------------
        # Production reliability function
        # ----------------------------------------------
        result = evaluate_prediction_reliability(
            rf_price=rf_price,
            lr_price=lr_price,
            hybrid_price=hybrid_price,
            comparable_count=comp_count,
            similarity=similarity,
        )

        reliability_scores.append(
            result["score"]
        )

        reliability_levels.append(
            result["level"]
        )

        reliability_warnings.append(
            result["warnings"]
        )

        model_disagreements.append(
            result["model_disagreement"]
        )

        comparable_counts.append(
            comp_count
        )

        similarities.append(
            similarity
        )

    except Exception as e:

        reliability_scores.append(np.nan)
        reliability_levels.append(None)
        reliability_warnings.append([str(e)])
        model_disagreements.append(np.nan)

        comparable_counts.append(np.nan)
        similarities.append(np.nan)

        failed_rows.append(
            (idx, repr(e))
        )

    if i % 500 == 0:
        elapsed = time.time() - start

        print(
            f"Processed {i:,}/{len(cal):,} "
            f"({elapsed:.1f}s)"
        )


# ------------------------------------------------------------
# 4. Store results
# ------------------------------------------------------------

cal["reliability_score"] = reliability_scores
cal["reliability_level"] = reliability_levels

cal["reliability_warnings"] = reliability_warnings

cal["model_disagreement"] = model_disagreements

cal["comparable_count"] = comparable_counts
cal["similarity"] = similarities


# ------------------------------------------------------------
# 5. Summary
# ------------------------------------------------------------

elapsed = time.time() - start

print("\n=== RELIABILITY BUILD SUMMARY ===")

print("Rows:", len(cal))

print(
    "Successful:",
    int(cal["reliability_score"].notna().sum())
)

print(
    "Failed:",
    len(failed_rows)
)

print(
    "Elapsed seconds:",
    round(elapsed, 1)
)


print("\n=== SCORE DISTRIBUTION ===")

print(
    cal["reliability_score"].describe(
        percentiles=[
            .10, .25, .50,
            .75, .90, .95, .99
        ]
    )
)


print("\n=== LEVEL DISTRIBUTION ===")

print(
    cal["reliability_level"]
    .value_counts(dropna=False)
)


print("\n=== COMPARABLE COUNT ===")

print(
    cal["comparable_count"].describe(
        percentiles=[
            .10, .25, .50,
            .75, .90, .95, .99
        ]
    )
)


print("\n=== SIMILARITY ===")

print(
    cal["similarity"].describe(
        percentiles=[
            .10, .25, .50,
            .75, .90, .95, .99
        ]
    )
)


if failed_rows:

    print("\n=== FIRST FAILED ROWS ===")

    for x in failed_rows[:10]:
        print(x)


print("\n=== CELL 6B COMPLETE ===")
print("cal now contains production reliability data.")
print("Production files untouched.")
print("Production models untouched.")
print("No retraining performed.")

# ============================================================
# CELL 6B.4 — VERIFY CORRECT find_comparable INPUT
# ============================================================

import numpy as np
import pandas as pd

print("=== VERIFY FIND_COMPARABLE FIX ===")

# query จาก Cell 6B.3 เป็น DataFrame 1 แถว
print("query type :", type(query))
print("query shape:", query.shape)

# ------------------------------------------------------------
# FIX:
# find_comparable expects one record, not a DataFrame
# ------------------------------------------------------------

query_row = query.iloc[0]

print("query_row type:", type(query_row))

# ------------------------------------------------------------
# Run production comparable function
# ------------------------------------------------------------

comp = find_comparable(
    query_row,
    market_reference
)

print("\n=== COMPARABLE RESULT ===")
print(comp)

# ------------------------------------------------------------
# Extract values
# ------------------------------------------------------------

rf_price = float(row["rf_prediction"])
lr_price = float(row["lr_prediction"])

comparable_price = comp.get(
    "comparable_price",
    np.nan
)

comparable_count = int(
    comp.get("comparable_count", 0)
)

similarity = comp.get(
    "similarity",
    None
)

match_level = comp.get(
    "match_level",
    None
)

# Hybrid value is irrelevant to current reliability scoring,
# but the production function requires the argument.
if (
    comparable_price is not None
    and pd.notna(comparable_price)
    and np.isfinite(float(comparable_price))
):
    hybrid_price = float(comparable_price)
else:
    hybrid_price = (rf_price + lr_price) / 2.0

# ------------------------------------------------------------
# Run production reliability function
# ------------------------------------------------------------

reliability = evaluate_prediction_reliability(
    rf_price=rf_price,
    lr_price=lr_price,
    hybrid_price=hybrid_price,
    comparable_count=comparable_count,
    similarity=similarity,
)

print("\n=== RELIABILITY RESULT ===")
print(reliability)

# ------------------------------------------------------------
# Sanity checks
# ------------------------------------------------------------

assert isinstance(comp, dict)
assert isinstance(reliability, dict)

assert "comparable_count" in comp
assert "similarity" in comp
assert "match_level" in comp

assert "score" in reliability
assert "level" in reliability
assert "model_disagreement" in reliability

print("\n=== SANITY CHECK ===")
print("RF prediction       :", rf_price)
print("LR prediction       :", lr_price)
print("Comparable price    :", comparable_price)
print("Comparable count    :", comparable_count)
print("Similarity          :", similarity)
print("Match level         :", match_level)
print("Reliability score   :", reliability["score"])
print("Reliability level   :", reliability["level"])
print(
    "Model disagreement :",
    reliability["model_disagreement"]
)

print("\n=== CELL 6B.4 COMPLETE ===")
print("Single-row production reliability pipeline PASSED.")
print("Production files untouched.")
print("Production models untouched.")
print("No retraining performed.")

In [ ]:
# ============================================================
# CELL 6B.5 — FROZEN V1 RELIABILITY AUDIT BASELINE
# Reconstruct historical V1 reliability for all audit rows
# ============================================================

import time
import numpy as np
import pandas as pd
# ------------------------------------------------------------
# Frozen Reliability V1 baseline
# Keep the original pre-V2 scoring logic for audit comparison.
# ------------------------------------------------------------

prepare_input = utils.prepare_input
find_comparable = utils.find_comparable


def evaluate_reliability_v1(
    rf_price,
    lr_price,
    hybrid_price,
    comparable_count=0,
    similarity=None,
):
    warnings = []
    score = 100

    # 1) Comparable coverage
    if comparable_count <= 1:
        score -= 35
        warnings.append(
            "ข้อมูลรถ Comparable มีน้อยมาก จึงควรใช้ผลประมาณด้วยความระมัดระวัง"
        )
    elif comparable_count < 5:
        score -= 20
        warnings.append(
            "ข้อมูลรถ Comparable ยังมีจำนวนค่อนข้างน้อย"
        )
    elif comparable_count < 10:
        score -= 10

    # 2) RF/LR disagreement
    try:
        rf = float(rf_price)
        lr = float(lr_price)

        denominator = max(abs(rf), abs(lr), 1.0)
        disagreement = abs(rf - lr) / denominator

        if disagreement >= 0.50:
            score -= 30
            warnings.append(
                "Random Forest และ Linear Regression ให้ผลต่างกันมาก"
            )
        elif disagreement >= 0.30:
            score -= 20
            warnings.append(
                "Random Forest และ Linear Regression ให้ผลต่างกันค่อนข้างมาก"
            )
        elif disagreement >= 0.15:
            score -= 10

    except Exception:
        disagreement = None

    # 3) Similarity
    if similarity is not None:
        try:
            similarity_value = float(similarity)

            if similarity_value < 0.30:
                score -= 20
                warnings.append(
                    "ความคล้ายกับรถอ้างอิงอยู่ในระดับต่ำ"
                )
            elif similarity_value < 0.40:
                score -= 10
                warnings.append(
                    "ความคล้ายกับรถอ้างอิงอยู่ในระดับค่อนข้างต่ำ"
                )
        except Exception:
            pass

    score = max(0, min(100, score))

    if score >= 80:
        level = "สูง"
    elif score >= 60:
        level = "ปานกลาง"
    else:
        level = "ต่ำ"

    if not warnings:
        warnings.append(
            "ไม่พบสัญญาณผิดปกติที่สำคัญจาก Guardrail"
        )

    return {
        "score": score,
        "level": level,
        "warnings": warnings,
        "model_disagreement": disagreement,
    }

print("=== FROZEN V1 RELIABILITY AUDIT BASELINE ===")

# ------------------------------------------------------------
# 1. Start from existing predictions
# ------------------------------------------------------------

cal = audit_pred.copy()

print("Rows:", len(cal))

# ------------------------------------------------------------
# 2. Prepare output columns
# ------------------------------------------------------------

cal["comparable_price"] = np.nan
cal["comparable_count"] = np.nan
cal["similarity"] = np.nan
cal["match_level"] = None

cal["reliability_score"] = np.nan
cal["reliability_level"] = None
cal["model_disagreement"] = np.nan
cal["reliability_warnings"] = None


# ------------------------------------------------------------
# 3. Run verified production pipeline
# ------------------------------------------------------------

failed_rows = []
successful = 0

start = time.time()

print("\n=== RUNNING 6,307 ROWS ===")

for n, (idx, row) in enumerate(cal.iterrows(), start=1):

    try:

        # --------------------------------------------
        # Build production feature row
        # --------------------------------------------

        query_df = prepare_input(
            brand=row["brand"],
            model=row["model"],
            sub_model=row.get("sub_model"),
            model_year=row["model_year"],
            mileage=row["mileage"],
            fuel_type=row.get("fuel_type"),
            transmission=row.get("transmission"),
            engine_size=row.get("engine_size"),
            body_type=row.get("body_type"),
            color=row.get("color"),
            province=row.get("province"),
            location=row.get("location"),
            seller_type=row.get("seller_type"),
            number_of_seats=row.get("number_of_seats"),
        )

        # IMPORTANT FIX:
        # find_comparable expects ONE record (Series),
        # not a 1-row DataFrame.
        query_row = query_df.iloc[0]

        # --------------------------------------------
        # Comparable calculation
        # --------------------------------------------

        comp = find_comparable(
            query_row,
            market_reference
        )

        comparable_price = comp.get(
            "comparable_price",
            np.nan
        )

        comparable_count = int(
            comp.get("comparable_count", 0)
        )

        similarity = comp.get(
            "similarity",
            None
        )

        match_level = comp.get(
            "match_level",
            None
        )

        # --------------------------------------------
        # Existing production model predictions
        # --------------------------------------------

        rf_price = float(row["rf_prediction"])
        lr_price = float(row["lr_prediction"])

        # evaluate_prediction_reliability currently
        # requires hybrid_price.
        if (
            comparable_price is not None
            and pd.notna(comparable_price)
            and np.isfinite(float(comparable_price))
        ):
            hybrid_price = float(comparable_price)

        else:
            hybrid_price = (
                rf_price + lr_price
            ) / 2.0

        # --------------------------------------------
        # Production reliability function
        # --------------------------------------------

        reliability = evaluate_reliability_v1(
            rf_price=rf_price,
            lr_price=lr_price,
            hybrid_price=hybrid_price,
            comparable_count=comparable_count,
            similarity=similarity,
        )

        # --------------------------------------------
        # Store results
        # --------------------------------------------

        cal.at[idx, "comparable_price"] = comparable_price
        cal.at[idx, "comparable_count"] = comparable_count
        cal.at[idx, "similarity"] = similarity
        cal.at[idx, "match_level"] = match_level

        cal.at[idx, "reliability_score"] = reliability.get(
            "score",
            np.nan
        )

        cal.at[idx, "reliability_level"] = reliability.get(
            "level",
            None
        )

        cal.at[idx, "model_disagreement"] = reliability.get(
            "model_disagreement",
            np.nan
        )

        cal.at[idx, "reliability_warnings"] = " | ".join(
            reliability.get("warnings", [])
        )

        successful += 1

    except Exception as e:

        failed_rows.append(
            (idx, repr(e))
        )

    # progress
    if n % 500 == 0:

        elapsed = time.time() - start

        print(
            f"Processed {n:,}/{len(cal):,} "
            f"({elapsed:.1f}s)"
        )


elapsed = time.time() - start


# ------------------------------------------------------------
# 4. Build summary
# ------------------------------------------------------------

print("\n=== BUILD SUMMARY ===")

print("Rows       :", len(cal))
print("Successful :", successful)
print("Failed     :", len(failed_rows))
print("Elapsed sec:", round(elapsed, 2))


# ------------------------------------------------------------
# 5. Reliability score
# ------------------------------------------------------------

print("\n=== RELIABILITY SCORE DISTRIBUTION ===")

print(
    cal["reliability_score"].describe(
        percentiles=[
            .10, .25, .50, .75,
            .90, .95, .99
        ]
    )
)


# ------------------------------------------------------------
# 6. Reliability level
# ------------------------------------------------------------

print("\n=== RELIABILITY LEVEL DISTRIBUTION ===")

print(
    cal["reliability_level"].value_counts(
        dropna=False
    )
)


# ------------------------------------------------------------
# 7. Comparable coverage
# ------------------------------------------------------------

print("\n=== COMPARABLE COUNT DISTRIBUTION ===")

print(
    cal["comparable_count"].describe(
        percentiles=[
            .10, .25, .50, .75,
            .90, .95, .99
        ]
    )
)

print(
    "\nComparable >= 5:",
    int((cal["comparable_count"] >= 5).sum()),
    "/",
    len(cal)
)

print(
    "Comparable < 5:",
    int((cal["comparable_count"] < 5).sum()),
    "/",
    len(cal)
)


# ------------------------------------------------------------
# 8. Similarity
# ------------------------------------------------------------

print("\n=== SIMILARITY DISTRIBUTION ===")

print(
    cal["similarity"].describe(
        percentiles=[
            .10, .25, .50, .75,
            .90, .95, .99
        ]
    )
)


# ------------------------------------------------------------
# 9. Match level
# ------------------------------------------------------------

print("\n=== MATCH LEVEL DISTRIBUTION ===")

print(
    cal["match_level"].value_counts(
        dropna=False
    )
)


# ------------------------------------------------------------
# 10. Model disagreement
# ------------------------------------------------------------

print("\n=== MODEL DISAGREEMENT DISTRIBUTION ===")

print(
    cal["model_disagreement"].describe(
        percentiles=[
            .10, .25, .50, .75,
            .90, .95, .99
        ]
    )
)


# ------------------------------------------------------------
# 11. Failures
# ------------------------------------------------------------

if failed_rows:

    print("\n=== FIRST FAILED ROWS ===")

    for item in failed_rows[:20]:
        print(item)


# ------------------------------------------------------------
# 12. Critical integrity assertions
# ------------------------------------------------------------

print("\n=== INTEGRITY CHECK ===")

assert len(cal) == len(audit_pred)

assert (
    cal["listing_id"].reset_index(drop=True)
    .equals(
        audit_pred["listing_id"].reset_index(drop=True)
    )
)

print("PASS: row count unchanged")
print("PASS: listing_id order unchanged")


# ------------------------------------------------------------
# 13. Finish
# ------------------------------------------------------------

print("\n=== CELL 6B.5 COMPLETE ===")
print("Historical V1 audit baseline build finished.")
print("cal now contains reliability outputs.")
print("Production files untouched.")
print("Production models untouched.")
print("No retraining performed.")

In [ ]:
# ============================================================
# CELL 6C — RELIABILITY CALIBRATION ANALYSIS
# Check whether production reliability score reflects
# actual prediction error.
#
# READ-ONLY AUDIT:
# - No production files modified
# - No model retraining
# - No metadata modified
# ============================================================

import numpy as np
import pandas as pd

print("=== RELIABILITY CALIBRATION ANALYSIS ===")

# ------------------------------------------------------------
# 1. Preconditions
# ------------------------------------------------------------

required_vars = ["cal"]

missing_vars = [
    name for name in required_vars
    if name not in globals()
]

if missing_vars:
    raise RuntimeError(
        "Missing variables from previous cells: "
        + ", ".join(missing_vars)
    )

if len(cal) == 0:
    raise RuntimeError("cal is empty")

required_cols = [
    "price",
    "rf_prediction",
    "lr_prediction",
    "reliability_score",
    "reliability_level",
    "comparable_count",
    "similarity",
    "model_disagreement",
]

missing_cols = [
    c for c in required_cols
    if c not in cal.columns
]

if missing_cols:
    raise RuntimeError(
        "Missing required columns in cal: "
        + ", ".join(missing_cols)
    )

print("Rows:", len(cal))
print("Required columns: PASS")


# ------------------------------------------------------------
# 2. Build analysis copy
# ------------------------------------------------------------

analysis = cal.copy()

numeric_cols = [
    "price",
    "rf_prediction",
    "lr_prediction",
    "reliability_score",
    "comparable_count",
    "similarity",
    "model_disagreement",
]

for col in numeric_cols:
    analysis[col] = pd.to_numeric(
        analysis[col],
        errors="coerce"
    )

# Only rows with valid target/predictions/reliability
valid_mask = (
    analysis["price"].notna()
    & analysis["rf_prediction"].notna()
    & analysis["lr_prediction"].notna()
    & analysis["reliability_score"].notna()
    & np.isfinite(analysis["price"])
    & np.isfinite(analysis["rf_prediction"])
    & np.isfinite(analysis["lr_prediction"])
    & np.isfinite(analysis["reliability_score"])
    & (analysis["price"] > 0)
)

analysis = analysis.loc[valid_mask].copy()

print("Valid analysis rows:", len(analysis))

if len(analysis) == 0:
    raise RuntimeError("No valid rows available for calibration analysis")


# ------------------------------------------------------------
# 3. Actual prediction errors
# ------------------------------------------------------------

analysis["rf_abs_error"] = np.abs(
    analysis["price"] - analysis["rf_prediction"]
)

analysis["lr_abs_error"] = np.abs(
    analysis["price"] - analysis["lr_prediction"]
)

# Percentage error
analysis["rf_ape_percent"] = (
    analysis["rf_abs_error"]
    / analysis["price"]
    * 100.0
)

analysis["lr_ape_percent"] = (
    analysis["lr_abs_error"]
    / analysis["price"]
    * 100.0
)

# Symmetric percentage error — more stable when actual price is small
analysis["rf_smape_percent"] = (
    200.0
    * analysis["rf_abs_error"]
    / (
        np.abs(analysis["price"])
        + np.abs(analysis["rf_prediction"])
    ).clip(lower=1.0)
)

analysis["lr_smape_percent"] = (
    200.0
    * analysis["lr_abs_error"]
    / (
        np.abs(analysis["price"])
        + np.abs(analysis["lr_prediction"])
    ).clip(lower=1.0)
)


# ------------------------------------------------------------
# 4. Reliability level summary
# ------------------------------------------------------------

level_order = ["ต่ำ", "ปานกลาง", "สูง"]

rows = []

for level in level_order:

    group = analysis[
        analysis["reliability_level"] == level
    ]

    if len(group) == 0:
        continue

    rows.append({
        "reliability_level": level,
        "cars": len(group),

        "score_mean":
            group["reliability_score"].mean(),

        "score_median":
            group["reliability_score"].median(),

        "RF_MAE":
            group["rf_abs_error"].mean(),

        "RF_Median_AE":
            group["rf_abs_error"].median(),

        "RF_Median_APE_percent":
            group["rf_ape_percent"].median(),

        "RF_sMAPE_percent":
            group["rf_smape_percent"].mean(),

        "LR_MAE":
            group["lr_abs_error"].mean(),

        "LR_Median_AE":
            group["lr_abs_error"].median(),

        "LR_Median_APE_percent":
            group["lr_ape_percent"].median(),

        "LR_sMAPE_percent":
            group["lr_smape_percent"].mean(),

        "comparable_mean":
            group["comparable_count"].mean(),

        "similarity_mean":
            group["similarity"].mean(),

        "disagreement_mean":
            group["model_disagreement"].mean(),
    })

reliability_level_summary = pd.DataFrame(rows)

print("\n=== ERROR BY RELIABILITY LEVEL ===")

display(
    reliability_level_summary.round({
        "score_mean": 2,
        "score_median": 2,
        "RF_MAE": 0,
        "RF_Median_AE": 0,
        "RF_Median_APE_percent": 2,
        "RF_sMAPE_percent": 2,
        "LR_MAE": 0,
        "LR_Median_AE": 0,
        "LR_Median_APE_percent": 2,
        "LR_sMAPE_percent": 2,
        "comparable_mean": 2,
        "similarity_mean": 3,
        "disagreement_mean": 3,
    })
)


# ------------------------------------------------------------
# 5. Reliability score bands
# ------------------------------------------------------------

analysis["score_band"] = pd.cut(
    analysis["reliability_score"],
    bins=[-np.inf, 39, 59, 79, 100],
    labels=[
        "0-39",
        "40-59",
        "60-79",
        "80-100",
    ],
    include_lowest=True,
)

score_band_summary = (
    analysis
    .groupby(
        "score_band",
        observed=True
    )
    .agg(
        cars=("price", "size"),

        score_mean=(
            "reliability_score",
            "mean"
        ),

        RF_MAE=(
            "rf_abs_error",
            "mean"
        ),

        RF_Median_AE=(
            "rf_abs_error",
            "median"
        ),

        RF_Median_APE_percent=(
            "rf_ape_percent",
            "median"
        ),

        RF_sMAPE_percent=(
            "rf_smape_percent",
            "mean"
        ),

        LR_MAE=(
            "lr_abs_error",
            "mean"
        ),

        LR_Median_AE=(
            "lr_abs_error",
            "median"
        ),

        LR_Median_APE_percent=(
            "lr_ape_percent",
            "median"
        ),

        LR_sMAPE_percent=(
            "lr_smape_percent",
            "mean"
        ),
    )
    .reset_index()
)

print("\n=== ERROR BY RELIABILITY SCORE BAND ===")

display(
    score_band_summary.round({
        "score_mean": 2,
        "RF_MAE": 0,
        "RF_Median_AE": 0,
        "RF_Median_APE_percent": 2,
        "RF_sMAPE_percent": 2,
        "LR_MAE": 0,
        "LR_Median_AE": 0,
        "LR_Median_APE_percent": 2,
        "LR_sMAPE_percent": 2,
    })
)


# ------------------------------------------------------------
# 6. Score vs actual error correlation
# ------------------------------------------------------------

print("\n=== SCORE VS ACTUAL ERROR CORRELATION ===")

correlation_results = {}

error_columns = [
    "rf_abs_error",
    "rf_ape_percent",
    "rf_smape_percent",
    "lr_abs_error",
    "lr_ape_percent",
    "lr_smape_percent",
]

for error_col in error_columns:

    temp = analysis[
        [
            "reliability_score",
            error_col,
        ]
    ].replace(
        [np.inf, -np.inf],
        np.nan
    ).dropna()

    if len(temp) < 3:
        pearson = np.nan
        spearman = np.nan

    else:
        pearson = temp[
            "reliability_score"
        ].corr(
            temp[error_col],
            method="pearson",
        )

        spearman = temp[
            "reliability_score"
        ].corr(
            temp[error_col],
            method="spearman",
        )

    correlation_results[error_col] = {
        "Pearson": pearson,
        "Spearman": spearman,
    }

correlation_table = (
    pd.DataFrame(correlation_results)
    .T
    .reset_index()
    .rename(
        columns={"index": "error_metric"}
    )
)

display(
    correlation_table.round(4)
)


# ------------------------------------------------------------
# 7. Guardrail components vs actual RF error
# ------------------------------------------------------------

print("\n=== GUARDRAIL COMPONENT CORRELATION ===")

component_cols = [
    "comparable_count",
    "similarity",
    "model_disagreement",
]

component_rows = []

for component in component_cols:

    temp = analysis[
        [
            component,
            "rf_abs_error",
            "rf_smape_percent",
        ]
    ].replace(
        [np.inf, -np.inf],
        np.nan
    ).dropna()

    if len(temp) < 3:
        continue

    component_rows.append({
        "component": component,

        "corr_RF_abs_error":
            temp[component].corr(
                temp["rf_abs_error"],
                method="spearman",
            ),

        "corr_RF_sMAPE":
            temp[component].corr(
                temp["rf_smape_percent"],
                method="spearman",
            ),
    })

component_correlation = pd.DataFrame(
    component_rows
)

display(
    component_correlation.round(4)
)


# ------------------------------------------------------------
# 8. Basic calibration direction checks
# ------------------------------------------------------------

print("\n=== CALIBRATION DIRECTION CHECK ===")

high = analysis[
    analysis["reliability_level"] == "สูง"
]

medium = analysis[
    analysis["reliability_level"] == "ปานกลาง"
]

low = analysis[
    analysis["reliability_level"] == "ต่ำ"
]

def med_error(df, col):
    if len(df) == 0:
        return np.nan
    return float(df[col].median())


rf_high = med_error(
    high,
    "rf_abs_error"
)

rf_medium = med_error(
    medium,
    "rf_abs_error"
)

rf_low = med_error(
    low,
    "rf_abs_error"
)

print(
    "RF Median AE — HIGH   :",
    round(rf_high, 2)
    if np.isfinite(rf_high)
    else "N/A"
)

print(
    "RF Median AE — MEDIUM :",
    round(rf_medium, 2)
    if np.isfinite(rf_medium)
    else "N/A"
)

print(
    "RF Median AE — LOW    :",
    round(rf_low, 2)
    if np.isfinite(rf_low)
    else "N/A"
)

if all(
    np.isfinite(x)
    for x in [
        rf_high,
        rf_medium,
        rf_low,
    ]
):

    if rf_high <= rf_medium <= rf_low:

        print(
            "PASS: RF actual error increases "
            "as reliability decreases."
        )

    else:

        print(
            "CHECK: Reliability levels are not "
            "perfectly monotonic with RF actual error."
        )


# ------------------------------------------------------------
# 9. Reliability counts
# ------------------------------------------------------------

print("\n=== RELIABILITY COVERAGE ===")

coverage = (
    analysis[
        "reliability_level"
    ]
    .value_counts(
        dropna=False
    )
)

print(coverage)

print(
    "\nRows analyzed:",
    len(analysis),
    "/",
    len(cal),
)


# ------------------------------------------------------------
# 10. Preserve analysis result
# ------------------------------------------------------------

calibration_analysis = analysis.copy()

print("\n=== CELL 6C COMPLETE ===")
print(
    "calibration_analysis shape:",
    calibration_analysis.shape,
)
print(
    "Production files untouched."
)
print(
    "Production models untouched."
)
print(
    "No retraining performed."
)
print(
    "Ready for Cell 6D."
)

In [ ]:
# ============================================================
# CELL 6D — RELIABILITY FAILURE DIAGNOSIS
# Goal:
#   1) Find HIGH reliability but HIGH actual RF error
#   2) Find LOW reliability but LOW actual RF error
#   3) Diagnose which guardrail components are responsible
# IMPORTANT:
#   - Audit only
#   - No production files modified
#   - No model retraining
# ============================================================

import numpy as np
import pandas as pd

print("=== RELIABILITY FAILURE DIAGNOSIS ===")

# ------------------------------------------------------------
# 1. Safety checks
# ------------------------------------------------------------

if "calibration_analysis" not in globals():
    raise RuntimeError(
        "ไม่พบ calibration_analysis — กรุณารัน Cell 6C ก่อน"
    )

diag = calibration_analysis.copy()

required_cols = [
    "reliability_score",
    "reliability_level",
    "comparable_count",
    "similarity",
    "model_disagreement",
    "rf_abs_error",
    "rf_ape_percent",
    "rf_smape_percent",
]

missing = [c for c in required_cols if c not in diag.columns]

if missing:
    raise RuntimeError(
        "Missing required columns: " + ", ".join(missing)
    )

print("Rows:", len(diag))
print("Required columns: PASS")


# ------------------------------------------------------------
# 2. Numeric cleanup
# ------------------------------------------------------------

numeric_cols = [
    "reliability_score",
    "comparable_count",
    "similarity",
    "model_disagreement",
    "rf_abs_error",
    "rf_ape_percent",
    "rf_smape_percent",
]

for col in numeric_cols:
    diag[col] = pd.to_numeric(diag[col], errors="coerce")

valid = diag[
    diag["reliability_score"].notna()
    & diag["rf_abs_error"].notna()
].copy()

print("Valid rows:", len(valid))


# ------------------------------------------------------------
# 3. Define actual-error thresholds from the audit population
# ------------------------------------------------------------

error_q25 = valid["rf_abs_error"].quantile(0.25)
error_q50 = valid["rf_abs_error"].quantile(0.50)
error_q75 = valid["rf_abs_error"].quantile(0.75)
error_q90 = valid["rf_abs_error"].quantile(0.90)

print("\n=== RF ABSOLUTE ERROR THRESHOLDS ===")
print(f"25th percentile : {error_q25:,.2f}")
print(f"Median          : {error_q50:,.2f}")
print(f"75th percentile : {error_q75:,.2f}")
print(f"90th percentile : {error_q90:,.2f}")


# ------------------------------------------------------------
# 4. Failure type A
# HIGH reliability but HIGH actual RF error
#
# High reliability = production level "สูง"
# High error       = top 25% actual RF error
# ------------------------------------------------------------

high_bad = valid[
    (valid["reliability_level"] == "สูง")
    & (valid["rf_abs_error"] >= error_q75)
].copy()

high_very_bad = valid[
    (valid["reliability_level"] == "สูง")
    & (valid["rf_abs_error"] >= error_q90)
].copy()

print("\n=== FAILURE A: HIGH RELIABILITY / HIGH RF ERROR ===")
print(
    "High reliability + error >= P75:",
    len(high_bad),
    "/",
    int((valid["reliability_level"] == "สูง").sum())
)

print(
    "High reliability + error >= P90:",
    len(high_very_bad),
    "/",
    int((valid["reliability_level"] == "สูง").sum())
)


# ------------------------------------------------------------
# 5. Failure type B
# LOW reliability but LOW actual RF error
#
# Low reliability = production level "ต่ำ"
# Low error        = bottom 25% actual RF error
# ------------------------------------------------------------

low_good = valid[
    (valid["reliability_level"] == "ต่ำ")
    & (valid["rf_abs_error"] <= error_q25)
].copy()

print("\n=== FAILURE B: LOW RELIABILITY / LOW RF ERROR ===")
print(
    "Low reliability + error <= P25:",
    len(low_good),
    "/",
    int((valid["reliability_level"] == "ต่ำ").sum())
)


# ------------------------------------------------------------
# 6. Build comparison groups
# ------------------------------------------------------------

high_all = valid[
    valid["reliability_level"] == "สูง"
].copy()

medium_all = valid[
    valid["reliability_level"] == "ปานกลาง"
].copy()

low_all = valid[
    valid["reliability_level"] == "ต่ำ"
].copy()


def summarize_group(name, df):
    if len(df) == 0:
        return {
            "group": name,
            "cars": 0,
            "score_mean": np.nan,
            "rf_mae": np.nan,
            "rf_median_ae": np.nan,
            "rf_median_smape": np.nan,
            "comparable_mean": np.nan,
            "similarity_mean": np.nan,
            "disagreement_mean": np.nan,
        }

    return {
        "group": name,
        "cars": len(df),

        "score_mean":
            df["reliability_score"].mean(),

        "rf_mae":
            df["rf_abs_error"].mean(),

        "rf_median_ae":
            df["rf_abs_error"].median(),

        "rf_median_smape":
            df["rf_smape_percent"].median(),

        "comparable_mean":
            df["comparable_count"].mean(),

        "similarity_mean":
            df["similarity"].mean(),

        "disagreement_mean":
            df["model_disagreement"].mean(),
    }


group_summary = pd.DataFrame([
    summarize_group("HIGH — all", high_all),
    summarize_group("HIGH — bad >= P75", high_bad),
    summarize_group("HIGH — very bad >= P90", high_very_bad),
    summarize_group("MEDIUM — all", medium_all),
    summarize_group("LOW — all", low_all),
    summarize_group("LOW — good <= P25", low_good),
])

print("\n=== FAILURE GROUP COMPARISON ===")
display(
    group_summary.round({
        "score_mean": 2,
        "rf_mae": 0,
        "rf_median_ae": 0,
        "rf_median_smape": 2,
        "comparable_mean": 2,
        "similarity_mean": 3,
        "disagreement_mean": 3,
    })
)


# ------------------------------------------------------------
# 7. Component bins
# ------------------------------------------------------------

valid["comparable_band"] = pd.cut(
    valid["comparable_count"],
    bins=[-np.inf, 1, 4, 9, np.inf],
    labels=[
        "0-1",
        "2-4",
        "5-9",
        "10+",
    ],
)

valid["similarity_band"] = pd.cut(
    valid["similarity"],
    bins=[-np.inf, 0.30, 0.40, 0.60, 0.80, np.inf],
    labels=[
        "<0.30",
        "0.30-0.39",
        "0.40-0.59",
        "0.60-0.79",
        "0.80+",
    ],
    right=False,
)

valid["disagreement_band"] = pd.cut(
    valid["model_disagreement"],
    bins=[-np.inf, 0.15, 0.30, 0.50, np.inf],
    labels=[
        "<0.15",
        "0.15-0.29",
        "0.30-0.49",
        "0.50+",
    ],
    right=False,
)


# ------------------------------------------------------------
# 8. Diagnostic function
# ------------------------------------------------------------

def component_diagnosis(df, component):

    result = (
        df.groupby(component, observed=False)
        .agg(
            cars=("rf_abs_error", "size"),
            score_mean=("reliability_score", "mean"),
            RF_MAE=("rf_abs_error", "mean"),
            RF_Median_AE=("rf_abs_error", "median"),
            RF_Median_sMAPE=("rf_smape_percent", "median"),
        )
        .reset_index()
    )

    return result


print("\n=== ERROR BY COMPARABLE COUNT BAND ===")
display(
    component_diagnosis(
        valid,
        "comparable_band"
    ).round(2)
)


print("\n=== ERROR BY SIMILARITY BAND ===")
display(
    component_diagnosis(
        valid,
        "similarity_band"
    ).round(2)
)


print("\n=== ERROR BY MODEL DISAGREEMENT BAND ===")
display(
    component_diagnosis(
        valid,
        "disagreement_band"
    ).round(2)
)


# ------------------------------------------------------------
# 9. Inspect worst HIGH-reliability cases
# ------------------------------------------------------------

candidate_cols = [
    "listing_id",
    "brand",
    "model",
    "sub_model",
    "model_year",
    "mileage",
    "price",
    "rf_prediction",
    "lr_prediction",
    "reliability_score",
    "reliability_level",
    "comparable_count",
    "similarity",
    "model_disagreement",
    "rf_abs_error",
    "rf_ape_percent",
    "rf_smape_percent",
]

show_cols = [
    c for c in candidate_cols
    if c in valid.columns
]

worst_high = (
    high_all
    .sort_values(
        "rf_abs_error",
        ascending=False
    )
    .head(20)
)

print("\n=== TOP 20 WORST ERRORS WITH HIGH RELIABILITY ===")

if len(worst_high):
    display(
        worst_high[show_cols]
        .reset_index(drop=True)
        .round(3)
    )
else:
    print("No rows found.")


# ------------------------------------------------------------
# 10. Inspect LOW reliability but surprisingly accurate
# ------------------------------------------------------------

best_low = (
    low_all
    .sort_values(
        "rf_abs_error",
        ascending=True
    )
    .head(20)
)

print("\n=== TOP 20 MOST ACCURATE WITH LOW RELIABILITY ===")

if len(best_low):
    display(
        best_low[show_cols]
        .reset_index(drop=True)
        .round(3)
    )
else:
    print("No rows found.")


# ------------------------------------------------------------
# 11. Quantify false-confidence rate
# ------------------------------------------------------------

n_high = len(high_all)

false_confidence_rate = (
    len(high_bad) / n_high * 100
    if n_high
    else np.nan
)

severe_false_confidence_rate = (
    len(high_very_bad) / n_high * 100
    if n_high
    else np.nan
)

n_low = len(low_all)

false_warning_rate = (
    len(low_good) / n_low * 100
    if n_low
    else np.nan
)

print("\n=== FAILURE RATES ===")

print(
    "False-confidence rate "
    "(HIGH reliability but RF error >= P75): "
    f"{false_confidence_rate:.2f}%"
)

print(
    "Severe false-confidence rate "
    "(HIGH reliability but RF error >= P90): "
    f"{severe_false_confidence_rate:.2f}%"
)

print(
    "False-warning rate "
    "(LOW reliability but RF error <= P25): "
    f"{false_warning_rate:.2f}%"
)


# ------------------------------------------------------------
# 12. Final diagnostic snapshot
# ------------------------------------------------------------

diagnostic_summary = {
    "rows": len(valid),

    "high_reliability_rows":
        len(high_all),

    "high_bad_p75_rows":
        len(high_bad),

    "high_very_bad_p90_rows":
        len(high_very_bad),

    "low_reliability_rows":
        len(low_all),

    "low_good_p25_rows":
        len(low_good),

    "false_confidence_rate_percent":
        false_confidence_rate,

    "severe_false_confidence_rate_percent":
        severe_false_confidence_rate,

    "false_warning_rate_percent":
        false_warning_rate,
}


print("\n=== DIAGNOSTIC SUMMARY ===")

for k, v in diagnostic_summary.items():

    if isinstance(v, float):
        print(f"{k}: {v:.3f}")
    else:
        print(f"{k}: {v}")


print("\n=== CELL 6D COMPLETE ===")
print("Production files untouched.")
print("Production models untouched.")
print("No retraining performed.")
print("Ready to decide whether reliability scoring needs recalibration.")

In [ ]:
# ============================================================
# CELL 6E — TARGET PRICE INTEGRITY AUDIT
#
# Diagnose suspicious target prices BEFORE recalibrating
# reliability.
#
# READ-ONLY:
# - No rows deleted from source data
# - No production files modified
# - No model modified
# - No retraining
# ============================================================

import numpy as np
import pandas as pd

print("=== TARGET PRICE INTEGRITY AUDIT ===")

# ------------------------------------------------------------
# 1. Preconditions
# ------------------------------------------------------------

if "calibration_analysis" not in globals():
    raise RuntimeError(
        "ไม่พบ calibration_analysis — กรุณารัน Cell 6C ก่อน"
    )

price_audit = calibration_analysis.copy()

required_cols = [
    "price",
    "rf_prediction",
    "lr_prediction",
    "reliability_score",
    "reliability_level",
    "rf_abs_error",
    "rf_smape_percent",
]

missing = [
    c for c in required_cols
    if c not in price_audit.columns
]

if missing:
    raise RuntimeError(
        "Missing required columns: " + ", ".join(missing)
    )

for col in [
    "price",
    "rf_prediction",
    "lr_prediction",
    "reliability_score",
    "rf_abs_error",
    "rf_smape_percent",
]:
    price_audit[col] = pd.to_numeric(
        price_audit[col],
        errors="coerce"
    )

print("Rows:", len(price_audit))
print("Required columns: PASS")


# ------------------------------------------------------------
# 2. Overall target distribution
# ------------------------------------------------------------

print("\n=== TARGET PRICE DISTRIBUTION ===")

print(
    price_audit["price"].describe(
        percentiles=[
            .001, .005, .01,
            .025, .05, .10,
            .25, .50, .75,
            .90, .95, .99,
            .995, .999,
        ]
    )
)


# ------------------------------------------------------------
# 3. Exact suspicious-price counts
# ------------------------------------------------------------

print("\n=== SUSPICIOUS PRICE COUNTS ===")

thresholds = [
    1,
    100,
    1_000,
    5_000,
    10_000,
    20_000,
    30_000,
    50_000,
    100_000,
]

threshold_rows = []

for threshold in thresholds:

    count = int(
        (price_audit["price"] <= threshold).sum()
    )

    threshold_rows.append({
        "price_lte": threshold,
        "cars": count,
        "percent": (
            count / len(price_audit) * 100
            if len(price_audit)
            else np.nan
        ),
    })

price_threshold_summary = pd.DataFrame(
    threshold_rows
)

display(
    price_threshold_summary.round({
        "percent": 3
    })
)


# ------------------------------------------------------------
# 4. Most common very-low prices
# ------------------------------------------------------------

print("\n=== MOST COMMON PRICES <= 100,000 ===")

low_price_counts = (
    price_audit.loc[
        price_audit["price"] <= 100_000,
        "price"
    ]
    .value_counts()
    .head(30)
    .rename_axis("price")
    .reset_index(name="cars")
)

display(low_price_counts)


# ------------------------------------------------------------
# 5. Inspect suspicious rows
# ------------------------------------------------------------

inspect_cols = [
    "listing_id",
    "brand",
    "model",
    "sub_model",
    "model_year",
    "mileage",
    "price",
    "rf_prediction",
    "lr_prediction",
    "reliability_score",
    "reliability_level",
    "comparable_count",
    "similarity",
    "model_disagreement",
    "rf_abs_error",
    "rf_smape_percent",
]

inspect_cols = [
    c for c in inspect_cols
    if c in price_audit.columns
]

suspicious_10k = (
    price_audit[
        price_audit["price"] <= 10_000
    ]
    .sort_values(
        ["price", "rf_abs_error"],
        ascending=[True, False]
    )
)

print("\n=== ROWS WITH PRICE <= 10,000 ===")
print("Count:", len(suspicious_10k))

display(
    suspicious_10k[
        inspect_cols
    ].head(50).reset_index(drop=True)
)


# ------------------------------------------------------------
# 6. Special inspection: price == 1
# ------------------------------------------------------------

price_one = price_audit[
    price_audit["price"] == 1
].copy()

print("\n=== PRICE == 1 INSPECTION ===")
print("Count:", len(price_one))

if len(price_one):

    display(
        price_one[
            inspect_cols
        ]
        .sort_values(
            "rf_abs_error",
            ascending=False
        )
        .head(50)
        .reset_index(drop=True)
    )


# ------------------------------------------------------------
# 7. Create diagnostic subsets ONLY
#
# IMPORTANT:
# These do NOT modify the production/test dataset.
# They exist only to measure sensitivity.
# ------------------------------------------------------------

diagnostic_thresholds = [
    1,
    10_000,
    30_000,
    50_000,
    100_000,
]

sensitivity_rows = []

for minimum_price in diagnostic_thresholds:

    subset = price_audit[
        price_audit["price"] > minimum_price
    ].copy()

    if len(subset) == 0:
        continue

    rf_mae = subset[
        "rf_abs_error"
    ].mean()

    rf_median_ae = subset[
        "rf_abs_error"
    ].median()

    rf_smape = subset[
        "rf_smape_percent"
    ].mean()

    # Correlation:
    # higher reliability should ideally mean lower error
    corr = (
        subset[
            [
                "reliability_score",
                "rf_smape_percent",
            ]
        ]
        .dropna()
        .corr(
            method="spearman"
        )
        .iloc[0, 1]
    )

    sensitivity_rows.append({
        "minimum_price_exclusive":
            minimum_price,

        "rows":
            len(subset),

        "rows_removed":
            len(price_audit) - len(subset),

        "RF_MAE":
            rf_mae,

        "RF_Median_AE":
            rf_median_ae,

        "RF_sMAPE":
            rf_smape,

        "Reliability_vs_RF_sMAPE_Spearman":
            corr,
    })


price_sensitivity = pd.DataFrame(
    sensitivity_rows
)

print("\n=== PRICE FILTER SENSITIVITY ===")

display(
    price_sensitivity.round({
        "RF_MAE": 0,
        "RF_Median_AE": 0,
        "RF_sMAPE": 2,
        "Reliability_vs_RF_sMAPE_Spearman": 4,
    })
)


# ------------------------------------------------------------
# 8. Re-check reliability levels under each
#    diagnostic target threshold
# ------------------------------------------------------------

level_rows = []

for minimum_price in diagnostic_thresholds:

    subset = price_audit[
        price_audit["price"] > minimum_price
    ].copy()

    for level in [
        "ต่ำ",
        "ปานกลาง",
        "สูง",
    ]:

        group = subset[
            subset["reliability_level"] == level
        ]

        if len(group) == 0:
            continue

        level_rows.append({
            "minimum_price_exclusive":
                minimum_price,

            "level":
                level,

            "cars":
                len(group),

            "RF_MAE":
                group["rf_abs_error"].mean(),

            "RF_Median_AE":
                group["rf_abs_error"].median(),

            "RF_Median_sMAPE":
                group["rf_smape_percent"].median(),
        })


price_level_sensitivity = pd.DataFrame(
    level_rows
)

print(
    "\n=== RELIABILITY LEVELS AFTER "
    "DIAGNOSTIC PRICE THRESHOLDS ==="
)

display(
    price_level_sensitivity.round({
        "RF_MAE": 0,
        "RF_Median_AE": 0,
        "RF_Median_sMAPE": 2,
    })
)


# ------------------------------------------------------------
# 9. Recalculate false-confidence sensitivity
# ------------------------------------------------------------

false_conf_rows = []

for minimum_price in diagnostic_thresholds:

    subset = price_audit[
        price_audit["price"] > minimum_price
    ].copy()

    if len(subset) == 0:
        continue

    p75 = subset[
        "rf_abs_error"
    ].quantile(0.75)

    p90 = subset[
        "rf_abs_error"
    ].quantile(0.90)

    high = subset[
        subset["reliability_level"] == "สูง"
    ]

    if len(high):

        high_bad = (
            high["rf_abs_error"] >= p75
        ).sum()

        high_severe = (
            high["rf_abs_error"] >= p90
        ).sum()

        false_rate = (
            high_bad / len(high) * 100
        )

        severe_rate = (
            high_severe / len(high) * 100
        )

    else:

        high_bad = 0
        high_severe = 0
        false_rate = np.nan
        severe_rate = np.nan

    false_conf_rows.append({
        "minimum_price_exclusive":
            minimum_price,

        "rows":
            len(subset),

        "high_reliability":
            len(high),

        "high_bad_P75":
            int(high_bad),

        "false_confidence_percent":
            false_rate,

        "high_bad_P90":
            int(high_severe),

        "severe_false_confidence_percent":
            severe_rate,
    })


false_confidence_sensitivity = pd.DataFrame(
    false_conf_rows
)

print(
    "\n=== FALSE-CONFIDENCE SENSITIVITY ==="
)

display(
    false_confidence_sensitivity.round({
        "false_confidence_percent": 2,
        "severe_false_confidence_percent": 2,
    })
)


# ------------------------------------------------------------
# 10. Preserve audit outputs
# ------------------------------------------------------------

target_price_audit = price_audit.copy()

print("\n=== CELL 6E COMPLETE ===")
print(
    "target_price_audit shape:",
    target_price_audit.shape
)
print("No source rows deleted.")
print("No production files modified.")
print("Production models untouched.")
print("No retraining performed.")
print(
    "Ready to determine whether target-price "
    "anomalies must be handled before reliability recalibration."
)

In [ ]:
# ============================================================
# CELL 6F — CLEAN CALIBRATION POPULATION + SIGNAL AUDIT
#
# Purpose:
# 1) Exclude confirmed placeholder target price == 1
#    from RELIABILITY CALIBRATION ONLY
# 2) Preserve all source rows
# 3) Measure which existing reliability signals actually
#    relate to RF prediction error
#
# READ-ONLY / OFFLINE:
# - No production files modified
# - No model modified
# - No retraining
# ============================================================

import numpy as np
import pandas as pd

print("=== CLEAN CALIBRATION POPULATION + SIGNAL AUDIT ===")

# ------------------------------------------------------------
# 1. Preconditions
# ------------------------------------------------------------

if "target_price_audit" in globals():
    base = target_price_audit.copy()

elif "calibration_analysis" in globals():
    base = calibration_analysis.copy()

else:
    raise RuntimeError(
        "ไม่พบ target_price_audit หรือ calibration_analysis "
        "— กรุณารัน Cell 6E ก่อน"
    )

required = [
    "price",
    "rf_prediction",
    "reliability_score",
    "reliability_level",
    "rf_abs_error",
    "rf_smape_percent",
    "comparable_count",
    "similarity",
    "model_disagreement",
]

missing = [c for c in required if c not in base.columns]

if missing:
    raise RuntimeError(
        "Missing required columns: " + ", ".join(missing)
    )

numeric_cols = [
    "price",
    "rf_prediction",
    "reliability_score",
    "rf_abs_error",
    "rf_smape_percent",
    "comparable_count",
    "similarity",
    "model_disagreement",
]

for col in numeric_cols:
    base[col] = pd.to_numeric(base[col], errors="coerce")


# ------------------------------------------------------------
# 2. Define calibration eligibility
#
# IMPORTANT:
# price == 1 is excluded ONLY from reliability calibration.
# Nothing is deleted from the source dataset.
# ------------------------------------------------------------

base["calibration_target_valid"] = (
    base["price"].notna()
    & np.isfinite(base["price"])
    & (base["price"] > 1)
)

excluded = base[~base["calibration_target_valid"]].copy()

clean_calibration = (
    base[base["calibration_target_valid"]]
    .copy()
    .reset_index(drop=True)
)

print("\n=== POPULATION ===")
print("Original rows :", len(base))
print("Eligible rows :", len(clean_calibration))
print("Excluded rows :", len(excluded))

if len(base):
    print(
        "Excluded %    :",
        round(len(excluded) / len(base) * 100, 3)
    )

print("\nExcluded price counts:")
print(
    excluded["price"]
    .value_counts(dropna=False)
    .head(20)
)


# ------------------------------------------------------------
# 3. Guardrail — verify we did NOT accidentally apply
#    a broad low-price cutoff
# ------------------------------------------------------------

valid_low_prices = clean_calibration[
    clean_calibration["price"] <= 100_000
]

print("\n=== LOW-PRICE GUARDRAIL ===")
print(
    "Valid rows retained with price <= 100,000:",
    len(valid_low_prices)
)

if len(valid_low_prices):
    print(
        "Minimum retained valid price:",
        valid_low_prices["price"].min()
    )

assert (clean_calibration["price"] > 1).all(), (
    "Clean calibration contains price <= 1"
)

print("Narrow exclusion rule: PASS")


# ------------------------------------------------------------
# 4. Baseline clean-population metrics
# ------------------------------------------------------------

print("\n=== CLEAN BASELINE ===")

baseline_metrics = pd.DataFrame([{
    "rows": len(clean_calibration),

    "RF_MAE":
        clean_calibration["rf_abs_error"].mean(),

    "RF_Median_AE":
        clean_calibration["rf_abs_error"].median(),

    "RF_sMAPE":
        clean_calibration["rf_smape_percent"].mean(),

    "RF_Median_sMAPE":
        clean_calibration["rf_smape_percent"].median(),
}])

display(
    baseline_metrics.round({
        "RF_MAE": 0,
        "RF_Median_AE": 0,
        "RF_sMAPE": 2,
        "RF_Median_sMAPE": 2,
    })
)


# ------------------------------------------------------------
# 5. Continuous signal correlations
#
# Desired directions:
#
# comparable_count ↑  -> error ↓
# similarity ↑        -> error ↓
# disagreement ↑      -> error ↑
# reliability ↑       -> error ↓
# ------------------------------------------------------------

print("\n=== SPEARMAN SIGNAL CORRELATIONS ===")

signal_cols = [
    "comparable_count",
    "similarity",
    "model_disagreement",
    "reliability_score",
]

corr_rows = []

for signal in signal_cols:

    tmp = (
        clean_calibration[
            [signal, "rf_abs_error", "rf_smape_percent"]
        ]
        .replace([np.inf, -np.inf], np.nan)
        .dropna()
    )

    if len(tmp) < 3:
        continue

    corr_rows.append({
        "signal": signal,

        "n": len(tmp),

        "vs_RF_abs_error":
            tmp[signal].corr(
                tmp["rf_abs_error"],
                method="spearman"
            ),

        "vs_RF_sMAPE":
            tmp[signal].corr(
                tmp["rf_smape_percent"],
                method="spearman"
            ),
    })

signal_correlations = pd.DataFrame(corr_rows)

display(
    signal_correlations.round({
        "vs_RF_abs_error": 4,
        "vs_RF_sMAPE": 4,
    })
)


# ------------------------------------------------------------
# 6. Helper for binned diagnostics
# ------------------------------------------------------------

def summarize_bins(df, bin_col, order=None):

    grouped = (
        df.groupby(
            bin_col,
            observed=False
        )
        .agg(
            cars=("rf_abs_error", "size"),
            RF_MAE=("rf_abs_error", "mean"),
            RF_Median_AE=("rf_abs_error", "median"),
            RF_Median_sMAPE=("rf_smape_percent", "median"),
            median_reliability=("reliability_score", "median"),
        )
        .reset_index()
    )

    if order is not None:
        grouped[bin_col] = pd.Categorical(
            grouped[bin_col],
            categories=order,
            ordered=True,
        )

        grouped = (
            grouped
            .sort_values(bin_col)
            .reset_index(drop=True)
        )

    return grouped


# ------------------------------------------------------------
# 7. Comparable-count audit
# ------------------------------------------------------------

print("\n=== COMPARABLE COUNT AUDIT ===")

comp_labels = [
    "0",
    "1",
    "2-3",
    "4-5",
    "6-9",
    "10+",
]

clean_calibration["comparable_bin_6f"] = pd.cut(
    clean_calibration["comparable_count"],
    bins=[
        -np.inf,
        0,
        1,
        3,
        5,
        9,
        np.inf,
    ],
    labels=comp_labels,
)

comparable_audit_6f = summarize_bins(
    clean_calibration,
    "comparable_bin_6f",
    comp_labels,
)

display(
    comparable_audit_6f.round({
        "RF_MAE": 0,
        "RF_Median_AE": 0,
        "RF_Median_sMAPE": 2,
        "median_reliability": 1,
    })
)


# ------------------------------------------------------------
# 8. Similarity audit
# ------------------------------------------------------------

print("\n=== SIMILARITY AUDIT ===")

sim_labels = [
    "<0.30",
    "0.30-0.49",
    "0.50-0.64",
    "0.65-0.79",
    "0.80+",
]

clean_calibration["similarity_bin_6f"] = pd.cut(
    clean_calibration["similarity"],
    bins=[
        -np.inf,
        0.30,
        0.50,
        0.65,
        0.80,
        np.inf,
    ],
    labels=sim_labels,
    right=False,
)

similarity_audit_6f = summarize_bins(
    clean_calibration,
    "similarity_bin_6f",
    sim_labels,
)

display(
    similarity_audit_6f.round({
        "RF_MAE": 0,
        "RF_Median_AE": 0,
        "RF_Median_sMAPE": 2,
        "median_reliability": 1,
    })
)


# ------------------------------------------------------------
# 9. Model-disagreement audit
# ------------------------------------------------------------

print("\n=== MODEL DISAGREEMENT AUDIT ===")

dis_labels = [
    "<0.10",
    "0.10-0.19",
    "0.20-0.29",
    "0.30-0.49",
    "0.50+",
]

clean_calibration["disagreement_bin_6f"] = pd.cut(
    clean_calibration["model_disagreement"],
    bins=[
        -np.inf,
        0.10,
        0.20,
        0.30,
        0.50,
        np.inf,
    ],
    labels=dis_labels,
    right=False,
)

disagreement_audit_6f = summarize_bins(
    clean_calibration,
    "disagreement_bin_6f",
    dis_labels,
)

display(
    disagreement_audit_6f.round({
        "RF_MAE": 0,
        "RF_Median_AE": 0,
        "RF_Median_sMAPE": 2,
        "median_reliability": 1,
    })
)


# ------------------------------------------------------------
# 10. Current reliability score audit
# ------------------------------------------------------------

print("\n=== CURRENT RELIABILITY SCORE AUDIT ===")

rel_labels = [
    "0-39",
    "40-49",
    "50-59",
    "60-69",
    "70-79",
    "80-89",
    "90-100",
]

clean_calibration["reliability_bin_6f"] = pd.cut(
    clean_calibration["reliability_score"],
    bins=[
        -np.inf,
        40,
        50,
        60,
        70,
        80,
        90,
        np.inf,
    ],
    labels=rel_labels,
    right=False,
)

reliability_score_audit_6f = summarize_bins(
    clean_calibration,
    "reliability_bin_6f",
    rel_labels,
)

display(
    reliability_score_audit_6f.round({
        "RF_MAE": 0,
        "RF_Median_AE": 0,
        "RF_Median_sMAPE": 2,
        "median_reliability": 1,
    })
)


# ------------------------------------------------------------
# 11. Reliability-level audit
# ------------------------------------------------------------

print("\n=== CURRENT RELIABILITY LEVEL AUDIT ===")

level_order = ["ต่ำ", "ปานกลาง", "สูง"]

reliability_level_audit_6f = (
    clean_calibration
    .groupby(
        "reliability_level",
        observed=False
    )
    .agg(
        cars=("rf_abs_error", "size"),
        RF_MAE=("rf_abs_error", "mean"),
        RF_Median_AE=("rf_abs_error", "median"),
        RF_Median_sMAPE=("rf_smape_percent", "median"),
        median_score=("reliability_score", "median"),
    )
    .reset_index()
)

reliability_level_audit_6f["reliability_level"] = (
    pd.Categorical(
        reliability_level_audit_6f["reliability_level"],
        categories=level_order,
        ordered=True,
    )
)

reliability_level_audit_6f = (
    reliability_level_audit_6f
    .sort_values("reliability_level")
    .reset_index(drop=True)
)

display(
    reliability_level_audit_6f.round({
        "RF_MAE": 0,
        "RF_Median_AE": 0,
        "RF_Median_sMAPE": 2,
        "median_score": 1,
    })
)


# ------------------------------------------------------------
# 12. High-reliability false-confidence profile
# ------------------------------------------------------------

print("\n=== HIGH-RELIABILITY FALSE-CONFIDENCE PROFILE ===")

p75_error = clean_calibration["rf_abs_error"].quantile(0.75)
p90_error = clean_calibration["rf_abs_error"].quantile(0.90)

high = clean_calibration[
    clean_calibration["reliability_level"] == "สูง"
].copy()

high["bad_P75"] = high["rf_abs_error"] >= p75_error
high["bad_P90"] = high["rf_abs_error"] >= p90_error

print("Clean P75 absolute error:", round(p75_error))
print("Clean P90 absolute error:", round(p90_error))
print("High-reliability rows:", len(high))

if len(high):

    print(
        "False confidence P75:",
        round(high["bad_P75"].mean() * 100, 2),
        "%"
    )

    print(
        "Severe false confidence P90:",
        round(high["bad_P90"].mean() * 100, 2),
        "%"
    )


# ------------------------------------------------------------
# 13. Diagnose characteristics of false-confidence rows
# ------------------------------------------------------------

false_confidence = high[
    high["bad_P75"]
].copy()

good_high = high[
    ~high["bad_P75"]
].copy()

comparison_rows = []

for name, group in [
    ("High reliability / GOOD", good_high),
    ("High reliability / FALSE CONFIDENCE", false_confidence),
]:

    if len(group) == 0:
        continue

    comparison_rows.append({
        "group": name,
        "cars": len(group),

        "median_abs_error":
            group["rf_abs_error"].median(),

        "median_sMAPE":
            group["rf_smape_percent"].median(),

        "median_comparable_count":
            group["comparable_count"].median(),

        "median_similarity":
            group["similarity"].median(),

        "median_model_disagreement":
            group["model_disagreement"].median(),

        "median_reliability":
            group["reliability_score"].median(),
    })

false_confidence_profile_6f = pd.DataFrame(
    comparison_rows
)

print("\n=== GOOD HIGH vs FALSE-CONFIDENCE HIGH ===")

display(
    false_confidence_profile_6f.round({
        "median_abs_error": 0,
        "median_sMAPE": 2,
        "median_comparable_count": 1,
        "median_similarity": 4,
        "median_model_disagreement": 4,
        "median_reliability": 1,
    })
)


# ------------------------------------------------------------
# 14. Worst clean high-reliability cases
# ------------------------------------------------------------

print("\n=== WORST CLEAN HIGH-RELIABILITY CASES ===")

case_cols = [
    "listing_id",
    "brand",
    "model",
    "sub_model",
    "model_year",
    "mileage",
    "price",
    "rf_prediction",
    "reliability_score",
    "comparable_count",
    "similarity",
    "model_disagreement",
    "rf_abs_error",
    "rf_smape_percent",
]

case_cols = [
    c for c in case_cols
    if c in high.columns
]

display(
    high[
        case_cols
    ]
    .sort_values(
        "rf_abs_error",
        ascending=False
    )
    .head(30)
    .reset_index(drop=True)
)


# ------------------------------------------------------------
# 15. Preserve outputs for Cell 6G
# ------------------------------------------------------------

reliability_calibration_clean = clean_calibration.copy()

print("\n=== CELL 6F COMPLETE ===")
print(
    "reliability_calibration_clean shape:",
    reliability_calibration_clean.shape
)

print(
    "Excluded placeholder-target rows:",
    len(excluded)
)

print("No source rows deleted.")
print("No production files modified.")
print("Production models untouched.")
print("No retraining performed.")
print(
    "Ready for Cell 6G — offline Reliability v2 candidate design."
)

In [ ]:
# ============================================================
# CELL 6G — OFFLINE RELIABILITY V2 CANDIDATE DESIGN
#
# Goal:
#   Build several Reliability v2 candidates OFFLINE and compare
#   their calibration against actual RF error.
#
# IMPORTANT:
#   - Does NOT modify utils.py
#   - Does NOT modify production scoring
#   - Does NOT modify models
#   - Does NOT retrain anything
# ============================================================

import numpy as np
import pandas as pd

print("=== OFFLINE RELIABILITY V2 CANDIDATE DESIGN ===")


# ------------------------------------------------------------
# 1. Preconditions
# ------------------------------------------------------------

if "reliability_calibration_clean" not in globals():
    raise RuntimeError(
        "ไม่พบ reliability_calibration_clean — กรุณารัน Cell 6F ก่อน"
    )

v2 = reliability_calibration_clean.copy()

required_cols = [
    "price",
    "rf_prediction",
    "rf_abs_error",
    "rf_smape_percent",
    "comparable_count",
    "similarity",
    "model_disagreement",
    "reliability_score",
    "reliability_level",
]

missing = [c for c in required_cols if c not in v2.columns]

if missing:
    raise RuntimeError(
        "Missing required columns: " + ", ".join(missing)
    )

numeric_cols = [
    "price",
    "rf_prediction",
    "rf_abs_error",
    "rf_smape_percent",
    "comparable_count",
    "similarity",
    "model_disagreement",
    "reliability_score",
]

for col in numeric_cols:
    v2[col] = pd.to_numeric(
        v2[col],
        errors="coerce"
    )

print("Rows:", len(v2))
print("Required columns: PASS")


# ------------------------------------------------------------
# 2. Normalize the three available reliability signals
#
# These are NOT learned from target error.
# They are deterministic transforms of production inputs.
# ------------------------------------------------------------

# Comparable coverage:
# 0 comps -> 0
# 10+ comps -> 1
v2["signal_comparable"] = (
    v2["comparable_count"]
    .fillna(0)
    .clip(lower=0, upper=10)
    / 10.0
)

# Similarity:
# already approximately 0..1
v2["signal_similarity"] = (
    v2["similarity"]
    .fillna(0)
    .clip(lower=0, upper=1)
)

# Agreement:
# disagreement 0   -> 1.0
# disagreement .5+ -> 0.0
#
# We keep this available for candidates that still
# want a small model-agreement contribution.
v2["signal_agreement"] = (
    1.0
    - (
        v2["model_disagreement"]
        .fillna(1.0)
        .clip(lower=0, upper=0.50)
        / 0.50
    )
).clip(0, 1)


# ------------------------------------------------------------
# 3. Candidate definitions
#
# V1 = current production score
#
# A = Similarity dominant + comparable + small agreement
# B = Similarity + comparable only
# C = Strong similarity emphasis
# D = Conservative / bottleneck style
#
# All scores are 0..100.
# ------------------------------------------------------------

v2["score_v1"] = (
    v2["reliability_score"]
    .clip(lower=0, upper=100)
)


# Candidate A
# 50% similarity
# 40% comparable coverage
# 10% model agreement
v2["score_v2_A"] = 100.0 * (
    0.50 * v2["signal_similarity"]
    + 0.40 * v2["signal_comparable"]
    + 0.10 * v2["signal_agreement"]
)


# Candidate B
# No model disagreement.
v2["score_v2_B"] = 100.0 * (
    0.60 * v2["signal_similarity"]
    + 0.40 * v2["signal_comparable"]
)


# Candidate C
# Similarity receives strongest weight.
v2["score_v2_C"] = 100.0 * (
    0.75 * v2["signal_similarity"]
    + 0.20 * v2["signal_comparable"]
    + 0.05 * v2["signal_agreement"]
)


# Candidate D
# Conservative:
# good score requires BOTH reasonable similarity
# and reasonable comparable coverage.
#
# geometric mean penalizes a weak component
# more strongly than a simple average.
v2["score_v2_D"] = 100.0 * np.sqrt(
    v2["signal_similarity"]
    * v2["signal_comparable"]
)


candidate_cols = [
    "score_v1",
    "score_v2_A",
    "score_v2_B",
    "score_v2_C",
    "score_v2_D",
]

for col in candidate_cols:
    v2[col] = (
        v2[col]
        .replace([np.inf, -np.inf], np.nan)
        .clip(lower=0, upper=100)
    )


# ------------------------------------------------------------
# 4. Basic score distributions
# ------------------------------------------------------------

print("\n=== CANDIDATE SCORE DISTRIBUTIONS ===")

score_distribution = (
    v2[candidate_cols]
    .describe(
        percentiles=[
            .10,
            .25,
            .50,
            .75,
            .90,
        ]
    )
    .T
)

display(
    score_distribution.round(2)
)


# ------------------------------------------------------------
# 5. Correlation with actual RF error
#
# Better reliability score should correlate NEGATIVELY
# with actual prediction error.
# ------------------------------------------------------------

print("\n=== CANDIDATE CORRELATION WITH RF ERROR ===")

corr_rows = []

for score_col in candidate_cols:

    tmp = (
        v2[
            [
                score_col,
                "rf_abs_error",
                "rf_smape_percent",
            ]
        ]
        .replace([np.inf, -np.inf], np.nan)
        .dropna()
    )

    if len(tmp) < 3:
        continue

    corr_rows.append({
        "candidate": score_col,

        "rows": len(tmp),

        "Spearman_abs_error":
            tmp[score_col].corr(
                tmp["rf_abs_error"],
                method="spearman"
            ),

        "Spearman_sMAPE":
            tmp[score_col].corr(
                tmp["rf_smape_percent"],
                method="spearman"
            ),

        "Pearson_sMAPE":
            tmp[score_col].corr(
                tmp["rf_smape_percent"],
                method="pearson"
            ),
    })


candidate_correlations_6g = pd.DataFrame(
    corr_rows
)

display(
    candidate_correlations_6g.round(4)
)


# ------------------------------------------------------------
# 6. Convert each candidate to provisional levels
#
# IMPORTANT:
# We intentionally use the SAME production boundaries:
#   HIGH   >= 80
#   MEDIUM >= 60
#   LOW    < 60
#
# This makes first comparison fair.
# Threshold optimization comes later in Cell 6I,
# only if justified.
# ------------------------------------------------------------

def score_to_level(score):

    if pd.isna(score):
        return np.nan

    if score >= 80:
        return "สูง"

    if score >= 60:
        return "ปานกลาง"

    return "ต่ำ"


for score_col in candidate_cols:

    level_col = score_col.replace(
        "score_",
        "level_"
    )

    v2[level_col] = (
        v2[score_col]
        .apply(score_to_level)
    )


# ------------------------------------------------------------
# 7. Level distribution
# ------------------------------------------------------------

print("\n=== CANDIDATE LEVEL DISTRIBUTIONS ===")

level_rows = []

for score_col in candidate_cols:

    level_col = score_col.replace(
        "score_",
        "level_"
    )

    counts = (
        v2[level_col]
        .value_counts()
    )

    level_rows.append({
        "candidate": score_col,

        "LOW":
            int(counts.get("ต่ำ", 0)),

        "MEDIUM":
            int(counts.get("ปานกลาง", 0)),

        "HIGH":
            int(counts.get("สูง", 0)),

        "HIGH_percent":
            (
                counts.get("สูง", 0)
                / len(v2)
                * 100
            ),
    })


candidate_level_distribution_6g = pd.DataFrame(
    level_rows
)

display(
    candidate_level_distribution_6g.round({
        "HIGH_percent": 2
    })
)


# ------------------------------------------------------------
# 8. Error by candidate level
# ------------------------------------------------------------

print("\n=== ERROR BY CANDIDATE LEVEL ===")

level_error_rows = []

level_order = [
    "ต่ำ",
    "ปานกลาง",
    "สูง",
]

for score_col in candidate_cols:

    level_col = score_col.replace(
        "score_",
        "level_"
    )

    for level in level_order:

        group = v2[
            v2[level_col] == level
        ]

        if len(group) == 0:
            continue

        level_error_rows.append({
            "candidate": score_col,
            "level": level,
            "cars": len(group),

            "score_median":
                group[score_col].median(),

            "RF_MAE":
                group["rf_abs_error"].mean(),

            "RF_Median_AE":
                group["rf_abs_error"].median(),

            "RF_Median_sMAPE":
                group["rf_smape_percent"].median(),

            "RF_mean_sMAPE":
                group["rf_smape_percent"].mean(),
        })


candidate_level_errors_6g = pd.DataFrame(
    level_error_rows
)

display(
    candidate_level_errors_6g.round({
        "score_median": 1,
        "RF_MAE": 0,
        "RF_Median_AE": 0,
        "RF_Median_sMAPE": 2,
        "RF_mean_sMAPE": 2,
    })
)


# ------------------------------------------------------------
# 9. Monotonicity check
#
# Desired:
# HIGH error < MEDIUM error < LOW error
#
# Check both Median AE and Median sMAPE.
# ------------------------------------------------------------

print("\n=== LEVEL MONOTONICITY CHECK ===")

monotonic_rows = []

for score_col in candidate_cols:

    level_col = score_col.replace(
        "score_",
        "level_"
    )

    stats = {}

    for level in level_order:

        group = v2[
            v2[level_col] == level
        ]

        stats[level] = {
            "median_ae":
                group["rf_abs_error"].median()
                if len(group)
                else np.nan,

            "median_smape":
                group["rf_smape_percent"].median()
                if len(group)
                else np.nan,
        }

    low_ae = stats["ต่ำ"]["median_ae"]
    med_ae = stats["ปานกลาง"]["median_ae"]
    high_ae = stats["สูง"]["median_ae"]

    low_smape = stats["ต่ำ"]["median_smape"]
    med_smape = stats["ปานกลาง"]["median_smape"]
    high_smape = stats["สูง"]["median_smape"]

    ae_monotonic = (
        np.isfinite(low_ae)
        and np.isfinite(med_ae)
        and np.isfinite(high_ae)
        and high_ae <= med_ae <= low_ae
    )

    smape_monotonic = (
        np.isfinite(low_smape)
        and np.isfinite(med_smape)
        and np.isfinite(high_smape)
        and high_smape <= med_smape <= low_smape
    )

    monotonic_rows.append({
        "candidate": score_col,

        "HIGH_Median_AE":
            high_ae,

        "MEDIUM_Median_AE":
            med_ae,

        "LOW_Median_AE":
            low_ae,

        "AE_monotonic":
            ae_monotonic,

        "HIGH_Median_sMAPE":
            high_smape,

        "MEDIUM_Median_sMAPE":
            med_smape,

        "LOW_Median_sMAPE":
            low_smape,

        "sMAPE_monotonic":
            smape_monotonic,
    })


candidate_monotonicity_6g = pd.DataFrame(
    monotonic_rows
)

display(
    candidate_monotonicity_6g.round({
        "HIGH_Median_AE": 0,
        "MEDIUM_Median_AE": 0,
        "LOW_Median_AE": 0,
        "HIGH_Median_sMAPE": 2,
        "MEDIUM_Median_sMAPE": 2,
        "LOW_Median_sMAPE": 2,
    })
)


# ------------------------------------------------------------
# 10. False-confidence comparison
#
# Definition stays identical to 6F:
# HIGH reliability but actual RF error >= population P75/P90.
# ------------------------------------------------------------

print("\n=== FALSE-CONFIDENCE COMPARISON ===")

p75_error = v2[
    "rf_abs_error"
].quantile(0.75)

p90_error = v2[
    "rf_abs_error"
].quantile(0.90)

false_conf_rows = []

for score_col in candidate_cols:

    level_col = score_col.replace(
        "score_",
        "level_"
    )

    high = v2[
        v2[level_col] == "สูง"
    ]

    n_high = len(high)

    if n_high:

        bad_p75 = int(
            (
                high["rf_abs_error"]
                >= p75_error
            ).sum()
        )

        bad_p90 = int(
            (
                high["rf_abs_error"]
                >= p90_error
            ).sum()
        )

        false_p75 = (
            bad_p75 / n_high * 100
        )

        false_p90 = (
            bad_p90 / n_high * 100
        )

        high_median_ae = (
            high["rf_abs_error"].median()
        )

        high_median_smape = (
            high["rf_smape_percent"].median()
        )

    else:

        bad_p75 = 0
        bad_p90 = 0
        false_p75 = np.nan
        false_p90 = np.nan
        high_median_ae = np.nan
        high_median_smape = np.nan

    false_conf_rows.append({
        "candidate": score_col,

        "HIGH_rows":
            n_high,

        "HIGH_percent":
            n_high / len(v2) * 100,

        "false_P75_rows":
            bad_p75,

        "false_confidence_P75_percent":
            false_p75,

        "false_P90_rows":
            bad_p90,

        "severe_false_confidence_P90_percent":
            false_p90,

        "HIGH_Median_AE":
            high_median_ae,

        "HIGH_Median_sMAPE":
            high_median_smape,
    })


candidate_false_confidence_6g = pd.DataFrame(
    false_conf_rows
)

display(
    candidate_false_confidence_6g.round({
        "HIGH_percent": 2,
        "false_confidence_P75_percent": 2,
        "severe_false_confidence_P90_percent": 2,
        "HIGH_Median_AE": 0,
        "HIGH_Median_sMAPE": 2,
    })
)


# ------------------------------------------------------------
# 11. Score-decile calibration
#
# A good continuous score should generally show lower error
# toward higher score deciles.
# ------------------------------------------------------------

print("\n=== SCORE DECILE CALIBRATION ===")

decile_outputs = {}

for score_col in candidate_cols:

    temp = v2[
        [
            score_col,
            "rf_abs_error",
            "rf_smape_percent",
        ]
    ].dropna().copy()

    # rank first to avoid qcut problems from duplicated scores
    temp["_rank"] = (
        temp[score_col]
        .rank(method="first")
    )

    temp["score_decile"] = pd.qcut(
        temp["_rank"],
        q=10,
        labels=False,
        duplicates="drop",
    ) + 1

    table = (
        temp.groupby(
            "score_decile",
            observed=True
        )
        .agg(
            cars=(score_col, "size"),

            score_median=(
                score_col,
                "median"
            ),

            RF_Median_AE=(
                "rf_abs_error",
                "median"
            ),

            RF_Median_sMAPE=(
                "rf_smape_percent",
                "median"
            ),
        )
        .reset_index()
    )

    decile_outputs[score_col] = table


# Compact display: only V1 and candidates
for score_col in candidate_cols:

    print(
        f"\n--- {score_col} ---"
    )

    display(
        decile_outputs[
            score_col
        ].round({
            "score_median": 2,
            "RF_Median_AE": 0,
            "RF_Median_sMAPE": 2,
        })
    )


# ------------------------------------------------------------
# 12. Build one compact comparison table
#
# IMPORTANT:
# This is descriptive only.
# Cell 6G does NOT automatically choose a winner.
# ------------------------------------------------------------

print("\n=== COMPACT CANDIDATE COMPARISON ===")

compact = (
    candidate_correlations_6g[
        [
            "candidate",
            "Spearman_abs_error",
            "Spearman_sMAPE",
        ]
    ]
    .merge(
        candidate_false_confidence_6g[
            [
                "candidate",
                "HIGH_rows",
                "HIGH_percent",
                "false_confidence_P75_percent",
                "severe_false_confidence_P90_percent",
                "HIGH_Median_AE",
                "HIGH_Median_sMAPE",
            ]
        ],
        on="candidate",
        how="left",
    )
    .merge(
        candidate_monotonicity_6g[
            [
                "candidate",
                "AE_monotonic",
                "sMAPE_monotonic",
            ]
        ],
        on="candidate",
        how="left",
    )
)

candidate_comparison_6g = compact.copy()

display(
    candidate_comparison_6g.round({
        "Spearman_abs_error": 4,
        "Spearman_sMAPE": 4,
        "HIGH_percent": 2,
        "false_confidence_P75_percent": 2,
        "severe_false_confidence_P90_percent": 2,
        "HIGH_Median_AE": 0,
        "HIGH_Median_sMAPE": 2,
    })
)


# ------------------------------------------------------------
# 13. Integrity checks
# ------------------------------------------------------------

assert len(v2) == len(
    reliability_calibration_clean
), "Row count changed unexpectedly"

assert (
    v2["listing_id"].reset_index(drop=True).equals(
        reliability_calibration_clean[
            "listing_id"
        ].reset_index(drop=True)
    )
), "listing_id order changed unexpectedly"

for col in candidate_cols:

    assert (
        v2[col]
        .dropna()
        .between(0, 100)
        .all()
    ), f"{col} contains score outside 0..100"


# ------------------------------------------------------------
# 14. Preserve outputs for Cell 6H
# ------------------------------------------------------------

reliability_v2_candidates = v2.copy()

print("\n=== CELL 6G COMPLETE ===")

print(
    "reliability_v2_candidates shape:",
    reliability_v2_candidates.shape
)

print(
    "Candidates evaluated:",
    ", ".join(candidate_cols)
)

print("Row integrity: PASS")
print("Score range integrity: PASS")

print("No production files modified.")
print("Audit did not modify the production reliability function.")
print("Current production status: Reliability V2 is already deployed.")
print("Production models untouched.")
print("No retraining performed.")

print(
    "Ready for Cell 6H — controlled V1 vs V2 candidate comparison."
)

In [ ]:
# ============================================================
# CELL 6H — V2 ROBUSTNESS + HIGH-THRESHOLD AUDIT
#
# Goal:
#   Test whether V2 improvements remain stable and determine
#   whether the production HIGH threshold (80) is appropriate.
#
# OFFLINE ONLY:
# - No utils.py modification
# - No production change
# - No retraining
# ============================================================

import numpy as np
import pandas as pd

print("=== V2 ROBUSTNESS + HIGH-THRESHOLD AUDIT ===")

# ------------------------------------------------------------
# 1. Preconditions
# ------------------------------------------------------------

if "reliability_v2_candidates" not in globals():
    raise RuntimeError(
        "ไม่พบ reliability_v2_candidates — กรุณารัน Cell 6G ก่อน"
    )

df = reliability_v2_candidates.copy()

candidate_cols = [
    "score_v1",
    "score_v2_A",
    "score_v2_B",
    "score_v2_C",
    "score_v2_D",
]

required = [
    "listing_id",
    "rf_abs_error",
    "rf_smape_percent",
] + candidate_cols

missing = [c for c in required if c not in df.columns]

if missing:
    raise RuntimeError(
        "Missing required columns: " + ", ".join(missing)
    )

print("Rows:", len(df))
print("Candidate columns: PASS")


# ------------------------------------------------------------
# 2. Fixed population error thresholds
# ------------------------------------------------------------

p75_error = df["rf_abs_error"].quantile(0.75)
p90_error = df["rf_abs_error"].quantile(0.90)

print("\n=== POPULATION ERROR THRESHOLDS ===")
print("P75 absolute error:", round(p75_error))
print("P90 absolute error:", round(p90_error))


# ------------------------------------------------------------
# 3. HIGH threshold sensitivity
#
# Test thresholds without changing candidate formulas.
# ------------------------------------------------------------

high_thresholds = [75, 80, 82.5, 85, 87.5, 90]

threshold_rows = []

for candidate in candidate_cols:

    for threshold in high_thresholds:

        high = df[df[candidate] >= threshold].copy()

        if len(high) == 0:
            continue

        bad75 = (
            high["rf_abs_error"] >= p75_error
        ).sum()

        bad90 = (
            high["rf_abs_error"] >= p90_error
        ).sum()

        threshold_rows.append({
            "candidate": candidate,
            "HIGH_threshold": threshold,

            "HIGH_rows": len(high),

            "HIGH_percent":
                len(high) / len(df) * 100,

            "false_P75_percent":
                bad75 / len(high) * 100,

            "severe_P90_percent":
                bad90 / len(high) * 100,

            "HIGH_Median_AE":
                high["rf_abs_error"].median(),

            "HIGH_Median_sMAPE":
                high["rf_smape_percent"].median(),

            "HIGH_Mean_sMAPE":
                high["rf_smape_percent"].mean(),
        })


threshold_audit_6h = pd.DataFrame(threshold_rows)

print("\n=== HIGH THRESHOLD SENSITIVITY ===")

display(
    threshold_audit_6h.round({
        "HIGH_threshold": 1,
        "HIGH_percent": 2,
        "false_P75_percent": 2,
        "severe_P90_percent": 2,
        "HIGH_Median_AE": 0,
        "HIGH_Median_sMAPE": 2,
        "HIGH_Mean_sMAPE": 2,
    })
)


# ------------------------------------------------------------
# 4. Compact threshold comparison for Candidate C
# ------------------------------------------------------------

print("\n=== CANDIDATE C — HIGH THRESHOLD DETAIL ===")

candidate_c_thresholds_6h = (
    threshold_audit_6h[
        threshold_audit_6h["candidate"] == "score_v2_C"
    ]
    .copy()
    .reset_index(drop=True)
)

display(
    candidate_c_thresholds_6h.round({
        "HIGH_threshold": 1,
        "HIGH_percent": 2,
        "false_P75_percent": 2,
        "severe_P90_percent": 2,
        "HIGH_Median_AE": 0,
        "HIGH_Median_sMAPE": 2,
        "HIGH_Mean_sMAPE": 2,
    })
)


# ------------------------------------------------------------
# 5. Bootstrap robustness
#
# Compare V1 vs V2 candidates over repeated resamples.
# We are checking whether correlation improvement is stable,
# rather than caused by a few unusual rows.
# ------------------------------------------------------------

print("\n=== BOOTSTRAP ROBUSTNESS ===")

rng = np.random.default_rng(42)

n_boot = 300
n_rows = len(df)

boot_records = []

for b in range(n_boot):

    sample_idx = rng.integers(
        0,
        n_rows,
        size=n_rows
    )

    sample = df.iloc[sample_idx]

    for candidate in candidate_cols:

        valid = (
            sample[
                [
                    candidate,
                    "rf_smape_percent"
                ]
            ]
            .replace([np.inf, -np.inf], np.nan)
            .dropna()
        )

        if len(valid) < 3:
            continue

        corr = valid[candidate].corr(
            valid["rf_smape_percent"],
            method="spearman"
        )

        boot_records.append({
            "bootstrap": b,
            "candidate": candidate,
            "spearman_smape": corr,
        })


bootstrap_raw_6h = pd.DataFrame(boot_records)

bootstrap_summary_rows = []

for candidate in candidate_cols:

    vals = (
        bootstrap_raw_6h.loc[
            bootstrap_raw_6h["candidate"] == candidate,
            "spearman_smape"
        ]
        .dropna()
    )

    bootstrap_summary_rows.append({
        "candidate": candidate,
        "boot_runs": len(vals),

        "mean":
            vals.mean(),

        "median":
            vals.median(),

        "CI_2.5":
            vals.quantile(0.025),

        "CI_97.5":
            vals.quantile(0.975),
    })


bootstrap_summary_6h = pd.DataFrame(
    bootstrap_summary_rows
)

display(
    bootstrap_summary_6h.round(4)
)


# ------------------------------------------------------------
# 6. Bootstrap direct improvement vs V1
#
# Negative delta = candidate has more negative correlation
# than V1 = desirable.
# ------------------------------------------------------------

print("\n=== BOOTSTRAP IMPROVEMENT VS V1 ===")

boot_pivot = (
    bootstrap_raw_6h
    .pivot(
        index="bootstrap",
        columns="candidate",
        values="spearman_smape"
    )
)

improvement_rows = []

for candidate in [
    "score_v2_A",
    "score_v2_B",
    "score_v2_C",
    "score_v2_D",
]:

    delta = (
        boot_pivot[candidate]
        - boot_pivot["score_v1"]
    ).dropna()

    improvement_rows.append({
        "candidate": candidate,

        "mean_delta_vs_v1":
            delta.mean(),

        "CI_2.5":
            delta.quantile(0.025),

        "CI_97.5":
            delta.quantile(0.975),

        "percent_better_than_v1":
            (delta < 0).mean() * 100,
    })


bootstrap_improvement_6h = pd.DataFrame(
    improvement_rows
)

display(
    bootstrap_improvement_6h.round({
        "mean_delta_vs_v1": 4,
        "CI_2.5": 4,
        "CI_97.5": 4,
        "percent_better_than_v1": 1,
    })
)


# ------------------------------------------------------------
# 7. Segment robustness
#
# Check common market segments if columns are available.
# This prevents choosing a score that works only in aggregate.
# ------------------------------------------------------------

print("\n=== SEGMENT ROBUSTNESS ===")

segment_cols = [
    c for c in [
        "brand",
        "model_year",
        "seller_type",
        "body_type",
    ]
    if c in df.columns
]

segment_records = []

# Brand: only sufficiently large brands
if "brand" in segment_cols:

    brand_counts = df["brand"].value_counts()

    major_brands = brand_counts[
        brand_counts >= 100
    ].index

    for brand in major_brands:

        group = df[df["brand"] == brand]

        for candidate in [
            "score_v1",
            "score_v2_A",
            "score_v2_B",
            "score_v2_C",
            "score_v2_D",
        ]:

            valid = (
                group[
                    [candidate, "rf_smape_percent"]
                ]
                .dropna()
            )

            if len(valid) < 30:
                continue

            segment_records.append({
                "segment_type": "brand",
                "segment": str(brand),
                "rows": len(valid),
                "candidate": candidate,

                "spearman_smape":
                    valid[candidate].corr(
                        valid["rf_smape_percent"],
                        method="spearman"
                    ),
            })


segment_robustness_6h = pd.DataFrame(
    segment_records
)

if len(segment_robustness_6h):

    # Compare median correlation across major brands
    segment_summary_6h = (
        segment_robustness_6h
        .groupby("candidate")
        .agg(
            segments=("segment", "nunique"),
            median_segment_corr=(
                "spearman_smape",
                "median"
            ),
            mean_segment_corr=(
                "spearman_smape",
                "mean"
            ),
            worst_segment_corr=(
                "spearman_smape",
                "max"
            ),
        )
        .reset_index()
    )

    display(
        segment_summary_6h.round(4)
    )

else:

    segment_summary_6h = pd.DataFrame()

    print(
        "No sufficiently large segment groups available."
    )


# ------------------------------------------------------------
# 8. Conservative shortlist
#
# NOTE:
# This is NOT an automatic production selection.
# It only filters candidates that satisfy basic evidence:
#
# - monotonic AE
# - monotonic sMAPE
# - stronger sMAPE correlation than V1
# ------------------------------------------------------------

print("\n=== EVIDENCE SHORTLIST ===")

if (
    "candidate_comparison_6g" not in globals()
):
    raise RuntimeError(
        "ไม่พบ candidate_comparison_6g — "
        "กรุณารัน Cell 6G ก่อน"
    )

comparison = candidate_comparison_6g.copy()

v1_corr = float(
    comparison.loc[
        comparison["candidate"] == "score_v1",
        "Spearman_sMAPE"
    ].iloc[0]
)

shortlist_6h = comparison[
    (comparison["candidate"] != "score_v1")
    & (comparison["AE_monotonic"] == True)
    & (comparison["sMAPE_monotonic"] == True)
    & (comparison["Spearman_sMAPE"] < v1_corr)
].copy()

# Add bootstrap evidence
shortlist_6h = shortlist_6h.merge(
    bootstrap_improvement_6h[
        [
            "candidate",
            "mean_delta_vs_v1",
            "CI_2.5",
            "CI_97.5",
            "percent_better_than_v1",
        ]
    ],
    on="candidate",
    how="left",
)

display(
    shortlist_6h.round({
        "Spearman_abs_error": 4,
        "Spearman_sMAPE": 4,
        "HIGH_percent": 2,
        "false_confidence_P75_percent": 2,
        "severe_false_confidence_P90_percent": 2,
        "HIGH_Median_AE": 0,
        "HIGH_Median_sMAPE": 2,
        "mean_delta_vs_v1": 4,
        "CI_2.5": 4,
        "CI_97.5": 4,
        "percent_better_than_v1": 1,
    })
)


# ------------------------------------------------------------
# 9. Integrity checks
# ------------------------------------------------------------

assert len(df) == 6193, (
    f"Unexpected row count: {len(df)}"
)

assert df["listing_id"].is_unique, (
    "listing_id is not unique"
)

assert len(bootstrap_summary_6h) == len(
    candidate_cols
)

print("\nIntegrity checks: PASS")


# ------------------------------------------------------------
# 10. Preserve for Cell 6I
# ------------------------------------------------------------

reliability_robustness_6h = df.copy()

print("\n=== CELL 6H COMPLETE ===")
print(
    "reliability_robustness_6h shape:",
    reliability_robustness_6h.shape
)

print("No production files modified.")
print("Production reliability function untouched.")
print("Production models untouched.")
print("No retraining performed.")

print(
    "Ready for Cell 6I — final offline formula + "
    "threshold decision and production-change specification."
)

In [ ]:
# ============================================================
# CELL 6I — FINAL RELIABILITY V2 VALIDATION SPECIFICATION
#
# Selected offline candidate:
#   score_v2_C
#
# Proposed level thresholds:
#   HIGH   >= 80
#   MEDIUM >= 60 and < 80
#   LOW    < 60
#
# IMPORTANT:
# - This cell DOES NOT modify utils.py
# - DOES NOT modify production
# - DOES NOT retrain models
# ============================================================

import numpy as np
import pandas as pd
import json

print("=== FINAL RELIABILITY V2 VALIDATION SPECIFICATION ===")

# ------------------------------------------------------------
# 1. Preconditions
# ------------------------------------------------------------

if "reliability_v2_candidates" not in globals():
    raise RuntimeError(
        "ไม่พบ reliability_v2_candidates — รัน Cell 6G ก่อน"
    )

if "threshold_audit_6h" not in globals():
    raise RuntimeError(
        "ไม่พบ threshold_audit_6h — รัน Cell 6H ก่อน"
    )

if "bootstrap_improvement_6h" not in globals():
    raise RuntimeError(
        "ไม่พบ bootstrap_improvement_6h — รัน Cell 6H ก่อน"
    )

df = reliability_v2_candidates.copy()

SELECTED_SCORE = "score_v2_C"
HIGH_THRESHOLD = 80.0
MEDIUM_THRESHOLD = 60.0

assert SELECTED_SCORE in df.columns

print("Selected candidate :", SELECTED_SCORE)
print("HIGH threshold     :", HIGH_THRESHOLD)
print("MEDIUM threshold   :", MEDIUM_THRESHOLD)
print("Rows               :", len(df))


# ------------------------------------------------------------
# 2. Create proposed V2 levels
# ------------------------------------------------------------

score = pd.to_numeric(
    df[SELECTED_SCORE],
    errors="coerce"
)

df["proposed_v2_score"] = score

df["proposed_v2_level"] = np.select(
    [
        score >= HIGH_THRESHOLD,
        score >= MEDIUM_THRESHOLD,
    ],
    [
        "สูง",
        "ปานกลาง",
    ],
    default="ต่ำ",
)

assert df["proposed_v2_score"].notna().all()
assert df["proposed_v2_score"].between(0, 100).all()


# ------------------------------------------------------------
# 3. Distribution
# ------------------------------------------------------------

print("\n=== PROPOSED V2 LEVEL DISTRIBUTION ===")

level_order = ["ต่ำ", "ปานกลาง", "สูง"]

distribution_rows = []

for level in level_order:

    mask = df["proposed_v2_level"] == level
    n = int(mask.sum())

    distribution_rows.append({
        "level": level,
        "cars": n,
        "percent": n / len(df) * 100,
        "score_median":
            df.loc[mask, "proposed_v2_score"].median()
            if n else np.nan,
    })

v2_level_distribution_6i = pd.DataFrame(
    distribution_rows
)

display(
    v2_level_distribution_6i.round({
        "percent": 2,
        "score_median": 2,
    })
)


# ------------------------------------------------------------
# 4. Error calibration by proposed level
# ------------------------------------------------------------

print("\n=== PROPOSED V2 ERROR CALIBRATION ===")

error_rows = []

for level in level_order:

    g = df[
        df["proposed_v2_level"] == level
    ].copy()

    if len(g) == 0:
        continue

    error_rows.append({
        "level": level,
        "cars": len(g),

        "score_median":
            g["proposed_v2_score"].median(),

        "RF_MAE":
            g["rf_abs_error"].mean(),

        "RF_Median_AE":
            g["rf_abs_error"].median(),

        "RF_Median_sMAPE":
            g["rf_smape_percent"].median(),

        "RF_Mean_sMAPE":
            g["rf_smape_percent"].mean(),
    })

v2_error_calibration_6i = pd.DataFrame(
    error_rows
)

display(
    v2_error_calibration_6i.round({
        "score_median": 2,
        "RF_MAE": 0,
        "RF_Median_AE": 0,
        "RF_Median_sMAPE": 2,
        "RF_Mean_sMAPE": 2,
    })
)


# ------------------------------------------------------------
# 5. Monotonicity verification
# ------------------------------------------------------------

print("\n=== MONOTONICITY VERIFICATION ===")

lookup = (
    v2_error_calibration_6i
    .set_index("level")
)

required_levels = ["สูง", "ปานกลาง", "ต่ำ"]

if not all(
    level in lookup.index
    for level in required_levels
):
    raise RuntimeError(
        "ระดับ reliability ไม่ครบ HIGH/MEDIUM/LOW"
    )

high_ae = lookup.loc["สูง", "RF_Median_AE"]
med_ae  = lookup.loc["ปานกลาง", "RF_Median_AE"]
low_ae  = lookup.loc["ต่ำ", "RF_Median_AE"]

high_smape = lookup.loc["สูง", "RF_Median_sMAPE"]
med_smape  = lookup.loc["ปานกลาง", "RF_Median_sMAPE"]
low_smape  = lookup.loc["ต่ำ", "RF_Median_sMAPE"]

ae_monotonic = (
    high_ae <= med_ae <= low_ae
)

smape_monotonic = (
    high_smape <= med_smape <= low_smape
)

print(
    "Median AE:",
    round(high_ae),
    "<=",
    round(med_ae),
    "<=",
    round(low_ae),
    "=>",
    ae_monotonic
)

print(
    "Median sMAPE:",
    round(high_smape, 2),
    "<=",
    round(med_smape, 2),
    "<=",
    round(low_smape, 2),
    "=>",
    smape_monotonic
)

assert ae_monotonic
assert smape_monotonic


# ------------------------------------------------------------
# 6. Compare directly with V1
# ------------------------------------------------------------

print("\n=== V1 vs PROPOSED V2 ===")

p75_error = df["rf_abs_error"].quantile(0.75)
p90_error = df["rf_abs_error"].quantile(0.90)

comparison_rows = []

for label, score_col in [
    ("V1", "score_v1"),
    ("V2_C", "proposed_v2_score"),
]:

    s = pd.to_numeric(
        df[score_col],
        errors="coerce"
    )

    high = df[s >= HIGH_THRESHOLD].copy()

    comparison_rows.append({
        "version": label,

        "HIGH_rows":
            len(high),

        "HIGH_percent":
            len(high) / len(df) * 100,

        "HIGH_Median_AE":
            high["rf_abs_error"].median(),

        "HIGH_Median_sMAPE":
            high["rf_smape_percent"].median(),

        "false_P75_percent":
            (
                high["rf_abs_error"] >= p75_error
            ).mean() * 100,

        "severe_P90_percent":
            (
                high["rf_abs_error"] >= p90_error
            ).mean() * 100,

        "Spearman_sMAPE":
            s.corr(
                df["rf_smape_percent"],
                method="spearman"
            ),
    })

v1_v2_final_comparison_6i = pd.DataFrame(
    comparison_rows
)

display(
    v1_v2_final_comparison_6i.round({
        "HIGH_percent": 2,
        "HIGH_Median_AE": 0,
        "HIGH_Median_sMAPE": 2,
        "false_P75_percent": 2,
        "severe_P90_percent": 2,
        "Spearman_sMAPE": 4,
    })
)


# ------------------------------------------------------------
# 7. Quantify improvement
# ------------------------------------------------------------

v1 = v1_v2_final_comparison_6i.iloc[0]
v2 = v1_v2_final_comparison_6i.iloc[1]

improvement_6i = {
    "HIGH_Median_AE_reduction_percent":
        (
            (v1["HIGH_Median_AE"]
             - v2["HIGH_Median_AE"])
            / v1["HIGH_Median_AE"]
            * 100
        ),

    "HIGH_Median_sMAPE_reduction_percent":
        (
            (v1["HIGH_Median_sMAPE"]
             - v2["HIGH_Median_sMAPE"])
            / v1["HIGH_Median_sMAPE"]
            * 100
        ),

    "false_P75_reduction_pp":
        (
            v1["false_P75_percent"]
            - v2["false_P75_percent"]
        ),

    "severe_P90_reduction_pp":
        (
            v1["severe_P90_percent"]
            - v2["severe_P90_percent"]
        ),
}

print("\n=== IMPROVEMENT VS V1 ===")

for k, v in improvement_6i.items():
    print(f"{k}: {v:.2f}")


# ------------------------------------------------------------
# 8. Bootstrap evidence
# ------------------------------------------------------------

boot_row = (
    bootstrap_improvement_6h[
        bootstrap_improvement_6h["candidate"]
        == SELECTED_SCORE
    ]
    .iloc[0]
)

print("\n=== BOOTSTRAP EVIDENCE ===")

print(
    "Mean Spearman delta vs V1:",
    round(
        float(boot_row["mean_delta_vs_v1"]),
        4
    )
)

print(
    "95% CI:",
    (
        round(float(boot_row["CI_2.5"]), 4),
        round(float(boot_row["CI_97.5"]), 4),
    )
)

print(
    "Bootstrap runs better than V1:",
    f"{boot_row['percent_better_than_v1']:.1f}%"
)


# ------------------------------------------------------------
# 9. Create implementation specification
#
# This is documentation only.
# Nothing is written into utils.py.
# ------------------------------------------------------------

reliability_v2_spec_6i = {
    "status": "VALIDATED_PRODUCTION_V2",

    "selected_candidate":
        SELECTED_SCORE,

    "level_thresholds": {
        "high": HIGH_THRESHOLD,
        "medium": MEDIUM_THRESHOLD,
    },

    "validation_population_rows":
        int(len(df)),

    "validation": {
        "median_AE_monotonic":
            bool(ae_monotonic),

        "median_sMAPE_monotonic":
            bool(smape_monotonic),

        "bootstrap_better_than_v1_percent":
            float(
                boot_row[
                    "percent_better_than_v1"
                ]
            ),

        "bootstrap_delta_95_CI": [
            float(boot_row["CI_2.5"]),
            float(boot_row["CI_97.5"]),
        ],

        "high_false_P75_percent":
            float(
                v2["false_P75_percent"]
            ),

        "high_severe_P90_percent":
            float(
                v2["severe_P90_percent"]
            ),
    },

    "production_change_applied_by_this_audit": False,
    "production_v2_status": "ALREADY_DEPLOYED",
    "models_retrained": False,
}

print("\n=== OFFLINE IMPLEMENTATION SPEC ===")
print(
    json.dumps(
        reliability_v2_spec_6i,
        indent=2,
        ensure_ascii=False
    )
)


# ------------------------------------------------------------
# 10. Final safety assertions
# ------------------------------------------------------------

assert len(df) == 6193
assert df["listing_id"].is_unique

assert (
    reliability_v2_spec_6i[
        "production_change_applied_by_this_audit"
    ]
    is False
)

assert (
    reliability_v2_spec_6i[
        "models_retrained"
    ]
    is False
)


# ------------------------------------------------------------
# 11. Preserve for final Cell 6J
# ------------------------------------------------------------

reliability_final_candidate_6i = df.copy()

print("\n=== CELL 6I COMPLETE ===")
print(
    "Selected offline candidate:",
    SELECTED_SCORE
)
print(
    "Proposed thresholds:",
    "LOW < 60 | MEDIUM 60-79.99 | HIGH >= 80"
)
print("Median AE monotonic:", ae_monotonic)
print("Median sMAPE monotonic:", smape_monotonic)

print("\nNo production files modified.")
print("Audit did not modify the production reliability function.")
print("Current production status: Reliability V2 is already deployed.")
print("Production models untouched.")
print("No retraining performed.")

print(
    "\nReady for Cell 6J — "
    "final audit report + production readiness gate."
)

In [ ]:
# ============================================================
# CELL 6J — FINAL RELIABILITY AUDIT REPORT + VALIDATION GATE
#
# FINAL CELL of 06_reliability_audit.ipynb
#
# Purpose:
# 1) Freeze final audit conclusions
# 2) Verify Reliability V2 evidence
# 3) Verify the tested formula is reproducible
# 4) Confirm whether the already-deployed V2 passes audit gates
#
# IMPORTANT:
# - Does NOT modify utils.py
# - Does NOT modify prediction_app.py
# - Does NOT modify model artifacts
# - Does NOT retrain anything
# ============================================================

import numpy as np
import pandas as pd
import json


print("=" * 72)
print("USED CAR AI — FINAL RELIABILITY AUDIT")
print("=" * 72)


# ------------------------------------------------------------
# 1. Preconditions
# ------------------------------------------------------------

required_objects = [
    "reliability_final_candidate_6i",
    "reliability_v2_spec_6i",
    "v1_v2_final_comparison_6i",
    "v2_error_calibration_6i",
    "bootstrap_improvement_6h",
]

missing_objects = [
    name
    for name in required_objects
    if name not in globals()
]

if missing_objects:
    raise RuntimeError(
        "Missing required objects: "
        + ", ".join(missing_objects)
    )


df = reliability_final_candidate_6i.copy()


print("\n=== 1. AUDIT POPULATION ===")
print("Calibration rows:", len(df))
print(
    "Unique listing_id:",
    df["listing_id"].nunique()
)


assert len(df) == 6193
assert df["listing_id"].is_unique

print("Population integrity: PASS")


# ------------------------------------------------------------
# 2. Freeze validated specification
# ------------------------------------------------------------

SELECTED_CANDIDATE = "score_v2_C"
HIGH_THRESHOLD = 80.0
MEDIUM_THRESHOLD = 60.0


print(
    "\n=== 2. VALIDATED RELIABILITY V2 SPECIFICATION ==="
)

print(
    "Candidate       :",
    SELECTED_CANDIDATE
)

print(
    "HIGH threshold  :",
    HIGH_THRESHOLD
)

print(
    "MEDIUM threshold:",
    MEDIUM_THRESHOLD
)


assert (
    reliability_v2_spec_6i[
        "selected_candidate"
    ]
    == SELECTED_CANDIDATE
)

assert (
    reliability_v2_spec_6i[
        "level_thresholds"
    ]["high"]
    == HIGH_THRESHOLD
)

assert (
    reliability_v2_spec_6i[
        "level_thresholds"
    ]["medium"]
    == MEDIUM_THRESHOLD
)

assert (
    reliability_v2_spec_6i[
        "production_change_applied_by_this_audit"
    ]
    is False
)

assert (
    reliability_v2_spec_6i[
        "production_v2_status"
    ]
    == "ALREADY_DEPLOYED"
)

assert (
    reliability_v2_spec_6i[
        "models_retrained"
    ]
    is False
)


print(
    "Production V2 status:",
    reliability_v2_spec_6i[
        "production_v2_status"
    ]
)


# ------------------------------------------------------------
# 3. Validated V2 formula
#
# Candidate C:
#
# score =
#   100 * (
#       0.75 * similarity_signal
#       + 0.20 * comparable_signal
#       + 0.05 * agreement_signal
#   )
#
# comparable_signal =
#   clip(comparable_count, 0, 10) / 10
#
# agreement_signal =
#   1 - clip(model_disagreement, 0, 0.50) / 0.50
#
# similarity_signal =
#   clip(similarity, 0, 1)
# ------------------------------------------------------------

print("\n=== 3. VALIDATED V2 FORMULA ===")


formula_spec = {
    "similarity_weight": 0.75,
    "comparable_weight": 0.20,
    "agreement_weight": 0.05,

    "comparable_cap": 10,
    "disagreement_cap": 0.50,

    "score_min": 0,
    "score_max": 100,

    "low_rule": "score < 60",
    "medium_rule": "60 <= score < 80",
    "high_rule": "score >= 80",
}


print(
    json.dumps(
        formula_spec,
        indent=2,
        ensure_ascii=False,
    )
)


# ------------------------------------------------------------
# 4. Recompute V2 independently
#
# This verifies that the documented formula exactly reproduces
# the formula evaluated in the previous audit cells.
# ------------------------------------------------------------

similarity = (
    pd.to_numeric(
        df["similarity"],
        errors="coerce",
    )
    .fillna(0)
    .clip(0, 1)
)


comparable_signal = (
    pd.to_numeric(
        df["comparable_count"],
        errors="coerce",
    )
    .fillna(0)
    .clip(0, 10)
    / 10.0
)


agreement_signal = (
    1.0
    - (
        pd.to_numeric(
            df["model_disagreement"],
            errors="coerce",
        )
        .fillna(1.0)
        .clip(0, 0.50)
        / 0.50
    )
).clip(0, 1)


score_recomputed = 100.0 * (
    0.75 * similarity
    + 0.20 * comparable_signal
    + 0.05 * agreement_signal
)

score_recomputed = (
    score_recomputed
    .clip(0, 100)
)


max_score_difference = np.max(
    np.abs(
        score_recomputed.to_numpy()
        - df[
            "proposed_v2_score"
        ].to_numpy()
    )
)


print(
    "\n=== 4. FORMULA REPRODUCIBILITY ==="
)

print(
    "Max difference vs Cell 6I:",
    max_score_difference,
)


assert max_score_difference < 1e-10

print("Formula reproducibility: PASS")


# ------------------------------------------------------------
# 5. Final calibration evidence
# ------------------------------------------------------------

print(
    "\n=== 5. FINAL LEVEL CALIBRATION ==="
)


display(
    v2_error_calibration_6i.round({
        "score_median": 2,
        "RF_MAE": 0,
        "RF_Median_AE": 0,
        "RF_Median_sMAPE": 2,
        "RF_Mean_sMAPE": 2,
    })
)


cal = (
    v2_error_calibration_6i
    .set_index("level")
)


ae_monotonic = (
    cal.loc[
        "สูง",
        "RF_Median_AE",
    ]
    <= cal.loc[
        "ปานกลาง",
        "RF_Median_AE",
    ]
    <= cal.loc[
        "ต่ำ",
        "RF_Median_AE",
    ]
)


smape_monotonic = (
    cal.loc[
        "สูง",
        "RF_Median_sMAPE",
    ]
    <= cal.loc[
        "ปานกลาง",
        "RF_Median_sMAPE",
    ]
    <= cal.loc[
        "ต่ำ",
        "RF_Median_sMAPE",
    ]
)


print(
    "Median AE monotonic   :",
    ae_monotonic,
)

print(
    "Median sMAPE monotonic:",
    smape_monotonic,
)


# ------------------------------------------------------------
# 6. V1 vs V2 final comparison
# ------------------------------------------------------------

print(
    "\n=== 6. V1 vs V2 FINAL COMPARISON ==="
)


display(
    v1_v2_final_comparison_6i.round({
        "HIGH_percent": 2,
        "HIGH_Median_AE": 0,
        "HIGH_Median_sMAPE": 2,
        "false_P75_percent": 2,
        "severe_P90_percent": 2,
        "Spearman_sMAPE": 4,
    })
)


v1_row = (
    v1_v2_final_comparison_6i[
        v1_v2_final_comparison_6i[
            "version"
        ]
        == "V1"
    ]
    .iloc[0]
)


v2_row = (
    v1_v2_final_comparison_6i[
        v1_v2_final_comparison_6i[
            "version"
        ]
        == "V2_C"
    ]
    .iloc[0]
)


# ------------------------------------------------------------
# 7. Bootstrap robustness
# ------------------------------------------------------------

print(
    "\n=== 7. BOOTSTRAP ROBUSTNESS ==="
)


boot = (
    bootstrap_improvement_6h[
        bootstrap_improvement_6h[
            "candidate"
        ]
        == SELECTED_CANDIDATE
    ]
    .iloc[0]
)


print(
    "Mean delta vs V1:",
    round(
        float(
            boot[
                "mean_delta_vs_v1"
            ]
        ),
        4,
    ),
)


print(
    "95% CI:",
    (
        round(
            float(
                boot["CI_2.5"]
            ),
            4,
        ),
        round(
            float(
                boot["CI_97.5"]
            ),
            4,
        ),
    ),
)


print(
    "Better than V1:",
    f"{float(boot['percent_better_than_v1']):.1f}%",
)


# ------------------------------------------------------------
# 8. Production validation gates
#
# Gates:
# - formula reproducible
# - Median AE monotonic
# - Median sMAPE monotonic
# - bootstrap improvement 100%
# - bootstrap CI entirely below zero
# - false confidence improves
# - severe false confidence improves
# - HIGH median AE improves
# - HIGH median sMAPE improves
# ------------------------------------------------------------

gates = {

    "formula_reproducible":
        bool(
            max_score_difference
            < 1e-10
        ),

    "median_AE_monotonic":
        bool(
            ae_monotonic
        ),

    "median_sMAPE_monotonic":
        bool(
            smape_monotonic
        ),

    "bootstrap_100pct_better":
        bool(
            float(
                boot[
                    "percent_better_than_v1"
                ]
            )
            == 100.0
        ),

    "bootstrap_CI_below_zero":
        bool(
            float(
                boot["CI_97.5"]
            )
            < 0
        ),

    "false_P75_improved":
        bool(
            v2_row[
                "false_P75_percent"
            ]
            < v1_row[
                "false_P75_percent"
            ]
        ),

    "severe_P90_improved":
        bool(
            v2_row[
                "severe_P90_percent"
            ]
            < v1_row[
                "severe_P90_percent"
            ]
        ),

    "HIGH_median_AE_improved":
        bool(
            v2_row[
                "HIGH_Median_AE"
            ]
            < v1_row[
                "HIGH_Median_AE"
            ]
        ),

    "HIGH_median_sMAPE_improved":
        bool(
            v2_row[
                "HIGH_Median_sMAPE"
            ]
            < v1_row[
                "HIGH_Median_sMAPE"
            ]
        ),
}


print(
    "\n=== 8. PRODUCTION VALIDATION GATES ==="
)


for gate, passed in gates.items():

    print(
        f"{gate:32s}:",
        "PASS"
        if passed
        else "FAIL",
    )


all_gates_passed = all(
    gates.values()
)


print(
    "\nAll gates passed:",
    all_gates_passed,
)


# ------------------------------------------------------------
# 9. Final validation status
#
# V2 is already deployed.
# This status describes whether the deployed V2 is supported
# by the completed offline reliability audit.
# ------------------------------------------------------------

if all_gates_passed:

    final_status = (
        "VALIDATED_PRODUCTION_V2"
    )

else:

    final_status = (
        "PRODUCTION_V2_REQUIRES_REVIEW"
    )


print(
    "\n=== 9. FINAL VALIDATION STATUS ==="
)

print(
    final_status
)


# ------------------------------------------------------------
# 10. Production validation specification
#
# This cell documents the validated production formula/status.
# It does NOT write anything to utils.py.
# ------------------------------------------------------------

production_change_spec_6j = {

    "audit_status":
        final_status,

    "function_target":
        "evaluate_prediction_reliability",

    "selected_formula":
        "Reliability V2 Candidate C",

    "formula": {

        "similarity_weight":
            0.75,

        "comparable_weight":
            0.20,

        "agreement_weight":
            0.05,

        "comparable_signal":
            (
                "clip(comparable_count, 0, 10) / 10"
            ),

        "agreement_signal":
            (
                "1 - clip(model_disagreement, "
                "0, 0.50) / 0.50"
            ),

        "similarity_signal":
            "clip(similarity, 0, 1)",

        "score":
            (
                "100 * "
                "(0.75*similarity_signal + "
                "0.20*comparable_signal + "
                "0.05*agreement_signal)"
            ),
    },

    "thresholds": {
        "high": 80,
        "medium": 60,
    },

    "validation_population": {

        "rows":
            int(len(df)),

        "placeholder_price_1_excluded":
            True,
    },

    "important_scope_note":
        (
            "price == 1 exclusion applies only to "
            "offline reliability calibration/evaluation; "
            "no source rows were deleted."
        ),

    "production_change_applied_by_this_audit":
        False,

    "production_v2_status":
        "ALREADY_DEPLOYED",

    "model_retraining_required":
        False,

    "models_retrained_by_this_audit":
        False,

    "next_step":
        (
            "Freeze the validated audit results and perform "
            "final regression/app smoke checks against the "
            "already-deployed Reliability V2."
        ),
}


print(
    "\n=== 10. PRODUCTION VALIDATION SPECIFICATION ==="
)


print(
    json.dumps(
        production_change_spec_6j,
        indent=2,
        ensure_ascii=False,
    )
)


# ------------------------------------------------------------
# 11. Final audit summary
# ------------------------------------------------------------

final_audit_summary_6j = pd.DataFrame([
    {

        "audit_population_rows":
            len(df),

        "selected_candidate":
            SELECTED_CANDIDATE,

        "HIGH_threshold":
            HIGH_THRESHOLD,

        "MEDIUM_threshold":
            MEDIUM_THRESHOLD,

        "V1_HIGH_percent":
            v1_row[
                "HIGH_percent"
            ],

        "V2_HIGH_percent":
            v2_row[
                "HIGH_percent"
            ],

        "V1_HIGH_Median_AE":
            v1_row[
                "HIGH_Median_AE"
            ],

        "V2_HIGH_Median_AE":
            v2_row[
                "HIGH_Median_AE"
            ],

        "V1_false_P75_percent":
            v1_row[
                "false_P75_percent"
            ],

        "V2_false_P75_percent":
            v2_row[
                "false_P75_percent"
            ],

        "V1_severe_P90_percent":
            v1_row[
                "severe_P90_percent"
            ],

        "V2_severe_P90_percent":
            v2_row[
                "severe_P90_percent"
            ],

        "V1_Spearman_sMAPE":
            v1_row[
                "Spearman_sMAPE"
            ],

        "V2_Spearman_sMAPE":
            v2_row[
                "Spearman_sMAPE"
            ],

        "all_validation_gates_passed":
            all_gates_passed,

        "production_v2_status":
            "ALREADY_DEPLOYED",

        "final_status":
            final_status,
    }
])


print(
    "\n=== 11. FINAL AUDIT SUMMARY ==="
)


display(
    final_audit_summary_6j.round({

        "V1_HIGH_percent":
            2,

        "V2_HIGH_percent":
            2,

        "V1_HIGH_Median_AE":
            0,

        "V2_HIGH_Median_AE":
            0,

        "V1_false_P75_percent":
            2,

        "V2_false_P75_percent":
            2,

        "V1_severe_P90_percent":
            2,

        "V2_severe_P90_percent":
            2,

        "V1_Spearman_sMAPE":
            4,

        "V2_Spearman_sMAPE":
            4,
    })
)


# ------------------------------------------------------------
# 12. Final assertions
# ------------------------------------------------------------

assert len(df) == 6193

assert df[
    "listing_id"
].is_unique

assert (
    max_score_difference
    < 1e-10
)

assert (
    production_change_spec_6j[
        "production_change_applied_by_this_audit"
    ]
    is False
)

assert (
    production_change_spec_6j[
        "production_v2_status"
    ]
    == "ALREADY_DEPLOYED"
)

assert (
    production_change_spec_6j[
        "model_retraining_required"
    ]
    is False
)

assert (
    production_change_spec_6j[
        "models_retrained_by_this_audit"
    ]
    is False
)


# ------------------------------------------------------------
# 13. Complete
# ------------------------------------------------------------

print(
    "\n"
    + "=" * 72
)

print(
    "CELL 6J COMPLETE — RELIABILITY AUDIT FINISHED"
)

print(
    "=" * 72
)


print(
    "\nFinal status:",
    final_status,
)


print("\nIMPORTANT:")

print(
    "- Reliability V2 is already deployed in production."
)

print(
    "- This audit did not modify utils.py."
)

print(
    "- This audit did not modify prediction_app.py."
)

print(
    "- RF/LR models were not modified."
)

print(
    "- No model retraining was performed."
)


if all_gates_passed:

    print(
        "\nAudit conclusion: deployed Reliability V2 "
        "is validated by the offline reliability audit."
    )

    print(
        "Next phase: final regression/app smoke checks "
        "and project finalization."
    )

else:

    print(
        "\nAudit conclusion: one or more validation "
        "gates failed."
    )

    print(
        "Next phase: review the deployed Reliability V2 "
        "before finalizing the project."
    )